<a href="https://colab.research.google.com/github/HyperTT/GeneticCF/blob/main/CS3960_FinalProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Counterfactuals and their affects

## Reading the Data

### Adult Data

Adult_Processed.csv

In [ ]:
!curl -L 'https://www.dropbox.com/scl/fi/3amctqg859jl6v31sehsu/adult_processed.csv?rlkey=in07prvvtanzwx0jscdnp0gt2&st=iopj86rd&dl=0' > "adult_processed.csv"

### Credit Data

Credit_Processed.csv

In [ ]:
!curl -L 'https://www.dropbox.com/scl/fi/0mlptssnzhn14vzkfitvr/credit_processed.csv?rlkey=935t68aa0olxxyazyt43yq8ws&st=bnp180ut&dl=0' > "credit_processed.csv"

### FICO Data




In [ ]:
!curl -L 'https://www.dropbox.com/scl/fi/iw35in4oy0pxlki3jxc62/heloc_dataset_v1.csv?rlkey=eo3mn88390xzw91om0dpv4rd7&st=j3p06f2i&dl=0' > 'heloc_dataset_v1'

## Viewing the Data

I will use Pandas for viewing the data.

In [ ]:
import pandas as pd
adultDF = pd.read_csv('adult_processed.csv')
creditDF = pd.read_csv('credit_processed.csv')
ficoDF = pd.read_csv('heloc_dataset_v1')

### Adult Data

When viewing the data frames it looks like some of the information was processed as digital choices rather than strings of data. For example, categorical data loses its meaning when there is not a system to convert numbers to categories. Non-categorical data that would be helpful to analyze would be `age`, `capital_gain`, `capital_loss`, and `hours_per_week`. I am unsure what the column `fnlwgt` represents.

I see that there is some sort of indexing difference here to account for Julia being a 1 indexing system vs python being a 0 indexing system. I prefer 0 indexing vs 1 indexes. Anyway the `sex` column has input values of 1s and 2s which is different from a true binary answer, so it makes me think that this column is accounting for people who are intersex or it may just be an error on the data compilation software / person in charge of creating this dataset.

In [ ]:
adultDF

In [ ]:
adultDF.describe()

In [ ]:
adultDF_90 = adultDF[adultDF['age'] == 90]
adultDF_90

In [ ]:
adultDF.info()

### Credit Data

The credit data has categorical data as well, and some of it was not converted to the correct type. The numerical data that will be helpful is

Last6MonthsData: This data will calculate historical trends for a person.

    `MaxBillAmountOverLast6Months`, `MaxPaymentAmountOverLast6Months`, `MonthsWithZeroBalanceOverLast6Months`, `MonthsQithLowSpendingOverLast6Months`, `MonthsWithHighSpendingOverLast6Months`

RecentData: This data will track current trends for different people.

    `MostRecentBillAmount`, `MostRecentPaymentAmount`

HistoricalData: This data will track all history of overdue payments.

    `TotalOverdueCounts`, `TotalMonthsOverDue`, `HasHistoryOfOverduePayments`

I am unsure how the overdue payment history is calculated but I am guessing that the total overdue counts is a column that directly affects the value of this column which I believe is 1 = True and 0 = False.

With how this relates to counter factuals, I am unsure if just having a history of overdue payments will affect the availiability of a loan. I will see what ways we can change the metrics to see if it allows for more or less harsh judgements.

In [ ]:
creditDF

In [ ]:
creditDF.describe()

In [ ]:
creditDF.info()

In [ ]:
creditDF['AgeGroup'] = creditDF['AgeGroup'].astype(int)
creditDF.info()

In [ ]:
creditDF_MaxBill = creditDF[creditDF['MaxBillAmountOverLast6Months'] == 50810.00]
creditDF_MaxBill

### Fico Data

Credit data and FICO data are similar because they both can affect a person's ability to apply for loans. The FICO dataset has several different columns and some categorical data that will be great to analyze. The `RiskPerformance` column I would like to research more about. I would like to see how the `ExternalRiskEstimate` is calculated and how this affects the `RiskPerformance` column. It looks like `RiskPerformance` is not solely determined by the `ExternalRiskEstimate` so this will take some more work to see if there is a relationship between the Good or Bad performance rating and the different values associated with the dataset.

There are more columns than Pandas is comfortable with showing. The dataframe has a ... in the middle so not all of the columns are being shown. To fix this, I added a setting to Pandas to show all columns.

I have checked my credit score, so I know some of the metrics that the credit companies look for when they are trying to see the risk value of a person. The biggest category is how often a person pays off their accounts, how many hard inquiries a person has, how many accounts a person has, how old are these accounts, etc. Something special to denote is that the older an account is, it will strengthen a credit score. The younger an account is, it will hinder a credit score. Hard inquieries will last up to 6 months before it will stop effecting a credit score.

Some of the Inquiry data includes these columns:

    `MSinceMostRecentInqexcl7days` or `Months Since Most Recent Inquiery Excluding the last 7 days`,
    `NumInqLast6M` or `Number of Inquieries in the Last 6 Months`,
    `NumInqLast6Mexcl7days` or `Number of Inquieries in the Last Six Months Excluding the last 7 Days`

Some of the Trading data includes these columns:

    `MSinceOldestTradeOpen` or `Months Since Oldest Trade Open`,
    `MSinceMostRecentTradeOpen` or `Months Since Most Recent Trade Open`,
    `AverageMInFile` or `Average Months In File`

Some of the Derogatory Remarks and Delinquencies will negatively affect the credit score:
Derogatory Remarks vs Satisfactory Trades

    `NumSatisfactoryTrades`,
    `NumTrades60Ever2DerogPubRecwill`,
    `NumTrades90Ever2DerogPubRec`

Delinquencies:

    `PercentTradesNeverDelq`,
    `MSinceMostRecentDelq`,
    `MaxDelq2PublicRecLast12M`,
    `MaxDelqEver`







In [ ]:
pd.set_option('display.max_columns', None)
ficoDF

When looking at the FICO data further using the describe method, it shows what the top 75% and the Maximum values for the external risk estimate. I would like to see which column has the risk of 94.

In [ ]:
ficoDF.describe()

In [ ]:
ficoDF.info()

It shows below that the person with the risk estimate of 94 is Bad performance. They never had a delinquecy as denoted by `PercentTradesNeverDelq` = 100% which does not make sense with the other columns like `MSinceMostRecentDelq` = -7 and `MaxDelq2PublicRecLast12M` = 7. It looks like if the `MaxDelqEver` is 8 and the total number of trades `NumTotalTrades` = 9 then they were delinquent on most of the accounts that they opened, which is not good. I do not know much about trades since credit cards and student loans are the most I have with the investment market. But when buying a house, all of these things start to become more important.

In [ ]:
ficoDF_94 = ficoDF[ficoDF['ExternalRiskEstimate'] == 94]
ficoDF_94

## Supplemental Code from Demo

### Ensuring Code Compatibility

##### scikit-learn compatibility

In [ ]:
!pip uninstall -y scikit-learn
# Version 1.0.2 is the most stable version from the 2021-2022 era
!pip install scikit-learn==1.0.2 --no-build-isolation

In [ ]:
!pip install skops

In [ ]:
import pickle
import sklearn.tree._tree as sklearn_tree

# 1. This 'monkey-patches' the Tree class to expect the 2021/2022 format
# but allows the 2024+ code to handle it without crashing.
if not hasattr(sklearn_tree, 'Tree'):
    # In some versions, the internal name is lowercase
    tree_class = sklearn_tree.Tree
else:
    tree_class = sklearn_tree.Tree

# 2. Use standard pickle to load your file
# We use a custom unpickler to handle potential namespace changes
class LegacyUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        # Handle cases where sklearn modules have been renamed since 2021
        if module == "sklearn.tree.tree":
            module = "sklearn.tree._tree"
        return super().find_class(module, name)

try:
    with open('compas_model.pickle', 'rb') as f:
        model = LegacyUnpickler(f).load()
    print("Success! Model loaded.")
except ValueError as e:
    print(f"Still hitting version error: {e}")
    print("Your model is likely too old for this patch. You must re-save it.")

In [ ]:
import pickle
import numpy as np
from sklearn.tree import DecisionTreeClassifier

# 1. Improved Stunt Double to handle any initialization arguments
class TreeStuntDouble:
    def __init__(self, *args, **kwargs):
        pass
    def __setstate__(self, state):
        self.state = state

# 2. Custom Unpickler to swap the real Tree for our Stunt Double
class LegacyLoader(pickle.Unpickler):
    def find_class(self, module, name):
        # Intercept the C-level Tree class before it validates the data
        if name == 'Tree' and 'sklearn.tree' in module:
            return TreeStuntDouble
        return super().find_class(module, name)

# 3. Load the file
with open('compas_model.pickle', 'rb') as f:
    model_obj = LegacyLoader(f).load()

# 4. Extract and fix the nodes
# If model_obj is the classifier, the tree is in .tree_
# If model_obj is the tree itself, use model_obj.state
target = model_obj.tree_ if hasattr(model_obj, 'tree_') else model_obj
raw_tree_state = target.state
nodes = raw_tree_state['nodes']

if 'missing_go_to_left' not in nodes.dtype.names:
    print("Detected 2021 format. Adding 'missing_go_to_left' field...")
    new_dtype = np.dtype(nodes.dtype.descr + [('missing_go_to_left', 'u1')])
    new_nodes = np.zeros(nodes.shape, dtype=new_dtype)
    for name in nodes.dtype.names:
        new_nodes[name] = nodes[name]
    raw_tree_state['nodes'] = new_nodes

# 5. Inject into a brand new model
# We must 'fit' it with 2 classes to match your original model's (507, 1, 2) shape
new_model = DecisionTreeClassifier()

# Dummy data: 10 rows, 1 feature, but 2 classes (0 and 1)
dummy_X = np.zeros((10, 1))
dummy_y = np.array([0, 1, 0, 1, 0, 1, 0, 1, 0, 1])

new_model.fit(dummy_X, dummy_y)

# Now the internal C-structures expect the (N, 1, 2) shape!
new_model.tree_.__setstate__(raw_tree_state)

print("Model successfully reconstructed!")


In [ ]:
import joblib

# Saving as a modern joblib file (preferred over pickle for sklearn)
joblib.dump(new_model, 'compas_model_2024.joblib')

print("Modern version saved as 'compas_model_2024.joblib'")

In [ ]:
import joblib
model = joblib.load('compas_model_2024.joblib')

In [ ]:
import sklearn
print(sklearn.__version__)

### Code For All Methods

##### compas_model.pickle

In [ ]:
import pickle

!wget -O compas_model.pickle "https://www.dropbox.com/scl/fi/xit9ao8egvhomzd1nqn4v/compas_model.pickle?rlkey=fiise52lj47r1irda0rl65jxz&st=zm2cobol&dl=0"

with open('compas_model.pickle', 'rb') as f:
    data = pickle.load(f)

##### credit_model.pickle

In [ ]:
import pickle

!wget -O credit_model.pickle "https://www.dropbox.com/scl/fi/b4crfqioxauoif20fn9ok/credit_model.pickle?rlkey=ves76yx8x3gppl2omufwgabdj&st=1c7yvmee&dl=0"

with open('credit_model.pickle', 'rb') as f:
    data = pickle.load(f)

##### adult_model.pickle

In [ ]:
import pickle

!wget -O adult_model.pickle "https://www.dropbox.com/scl/fi/20e819w5yqohi7bevoehi/adult_model.pickle?rlkey=wipvunakgogdy5wjzll8u1c25&st=blijzvjl&dl=0"

with open('adult_model.pickle', 'rb') as f:
    data = pickle.load(f)

##### adult_model_old.pickle

In [ ]:
import pickle

!wget -O adult_model_old.pickle "https://www.dropbox.com/scl/fi/mdbwes8y53ovpy83u9rls/adult_model_old.pickle?rlkey=fiq71cb7zurqtamhpmqa802x9&st=jmkn9ljv&dl=0"

with open('adult_model_old.pickle', 'rb') as f:
    data = pickle.load(f)

#### GeneticCF.jl

In [ ]:
import pandas as pd
import numpy as np
import time
import random
import copy
from typing import List, Set, Any, Tuple

class GeCo:
    @staticmethod
    def score(classifier, samples, desired_class, extra_col=0):
        # GeCo.score which evaluates samples against a classifier
        # classifier.predict_proba
        if hasattr(classifier, "predict_proba"):
            preds = classifier.predict_proba(samples)
            # Return probability of the desired class
            return preds[:, desired_class]
        else:
            # Fallback for classifiers that only have predict
            return (classifier.predict(samples) == desired_class).astype(float)

    @staticmethod
    def explain(orig_instance, data, plaf_program, classifier, desired_class=1):
        # Returns a list of explanations (often counterfactuals or actions)
        return [], None

class PLAFProgram:
    def __init__(self, groups: List[List[str]]):
        self.groups = groups

# a feature must be in exactly one group
class RuleFeatureGroup:
    def __init__(self, features: List[str], indexes: List[int], categorical: bool):
        self.features = features
        self.indexes = indexes
        self.categorical = categorical

def initRuleFeatureGroups(prog: PLAFProgram, data: pd.DataFrame):
    groups = []
    # Initialize with -1 (equivalent to undef in Julia)
    fidx_groupidx = np.full(data.shape[1], -1, dtype=np.int32)

    includedFeatures = set()

    for group in prog.groups:
        indexes = []
        for feature in group:
            assert feature in data.columns, f"The feature {feature} does not occur in the input data."
            assert feature not in includedFeatures, "Each feature can be in at most one group!"
            includedFeatures.add(feature)
            # findfirst(isequal(feature), propertynames(data)) -> get_loc (0-indexed)
            indexes.append(data.columns.get_loc(feature))

        groups.append(
            RuleFeatureGroup(
                list(group),
                indexes,
                True
            )
        )

    for feature in data.columns:
        if feature not in includedFeatures:
            assert feature in data.columns, f"The feature {feature} does not occur in the input data."
            indexes = [data.columns.get_loc(feature)]

            # elscitype(data[!,feature]) == Multiclass
            # check if the column is categorical or object
            is_categorical = pd.api.types.is_categorical_dtype(data[feature]) or pd.api.types.is_object_dtype(data[feature])

            groups.append(
                RuleFeatureGroup(
                    [feature],
                    indexes,
                    is_categorical
                )
            )

    for gidx, group in enumerate(groups):
        for fidx in group.indexes:
            fidx_groupidx[fidx] = gidx

    return groups, fidx_groupidx


class RuleComponent:
    def __init__(self, group: RuleFeatureGroup, direction: bool, index: int, group_index: int):
        self.group = group
        self.direction = direction
        self.index = index
        self.group_index = group_index

def sort_components(component: RuleComponent):
    return component.index

def initailComponents(groups: List[RuleFeatureGroup]):
    components = []
    for idx, group in enumerate(groups):
        # Julia 1-based indexing 2*idx-1 and 2*idx becomes 2*idx and 2*idx+1 in Python
        components.append(RuleComponent(group, True, 2 * idx, idx))
        components.append(RuleComponent(group, False, 2 * idx + 1, idx))
    return components


# the rule generated
class Rule:
    def __init__(self, component_num: int):
        self.rule_components: List[RuleComponent] = []
        # bit_representation: check whether two rule are the same
        self.bit_representation = np.zeros(component_num, dtype=bool)
        self.score: float = 0.0
        self.precision: float = 0.0
        self.generation: int = 0
        self.geco_verified: bool = False
        self.geco_generated: bool = False

def initialRule(component_num) -> Rule:
    return Rule(component_num)

def print_rule(orig_instance: pd.Series, rule: Rule):
    print(f"The precision of the rule is: {rule.precision}")
    print(f"This rule has {len(rule.rule_components)} components")

    included_groups = set()
    index = 1

    for rule_component in rule.rule_components:
        if rule_component.group in included_groups:
            # this group has already been printed
            continue

        # check whether the other direction also in the group
        bidirection = False
        for rule_component_other in rule.rule_components:
            if rule_component_other.group_index == rule_component.group_index and rule_component_other.direction != rule_component.direction:
                bidirection = True
                break

        if bidirection:
            if rule_component.direction:
                continue
            print(f"\t Component {index} and {index + 1} defines exact match on the feature group:")
            index += 1
        elif rule_component.direction: # >=
            print(f"\t Component {index} defines lower bound on the feature group:")
        else:
            print(f"\t Component {index} defines upper bound on the feature group:")

        for feature in rule_component.group.features:
            if bidirection:
                print(f"\t\t For feature {feature}: x = {orig_instance[feature]}")
            elif rule_component.direction: # >=
                print(f"\t\t For feature {feature}: x >= {orig_instance[feature]}")
            else:
                print(f"\t\t For feature {feature}: x <= {orig_instance[feature]}")

        index += 1
        included_groups.add(rule_component.group)

class ActionComponent:
    def __init__(self, index: int, direction: int):
        self.index = index
        self.direction = direction # 1 -> become larger

class Action:
    def __init__(self, components: List[ActionComponent]):
        self.components = components

def sort_action_cardinality(action: Action):
    return len(action.components)

def check_converge(population: List[Rule], generation: int, geco_mutation: bool) -> bool:
    # Minimal convergence logic (as it was missing in original snippet)
    if len(population) == 0:
        return False
    return generation > 5 and population[0].score > 0.99

def generate_rules(orig_instance: pd.Series, data: pd.DataFrame, classifier, plaf_program: PLAFProgram,
                   epslin: float = 0.0,
                   max_num_population: int = 50,
                   geco_initial: bool = False,
                   geco_mutation: bool = False,
                   desired_class: int = 1,
                   max_generation: int = 50,
                   sample_num: int = 1000,
                   geco_check_size: int = 5,
                   reduction = False,
                   ablation: bool = False):

    if not ablation:
        groups, fidx_gidx = initRuleFeatureGroups(plaf_program, data)
        components = initailComponents(groups)

        population, checked_set = [], set()

        population, population_bit_sets = initialPopulation_naive(components)
        #  tuples for the checked_set
        checked_set = {tuple(b) for b in population_bit_sets}

        if geco_initial:
            population, checked_set = initialPopulationWithGeco(data, orig_instance, plaf_program, classifier, fidx_gidx, components, population, checked_set, desired_class = desired_class)

        space = initialSpace(orig_instance, data)
        selection_inplace(population, data, orig_instance, space, classifier, checked_set, pop_size = max_num_population, epslin = epslin, desired_class = desired_class, sample_num = sample_num)

        generation = 0
        converge = False

        geco_checked = True
        if epslin == 0.0 and geco_mutation:
            geco_checked = False

        last_geco = 0

        while generation < max_generation and (not converge or len(population) < 1 or population[0].precision < 1 - epslin):
            generation += 1

            if geco_mutation and generation - last_geco > 3:
                last_geco = generation
                mutation_geco_inplace(population, checked_set, data, orig_instance, classifier, plaf_program, fidx_gidx, components, generation, desired_class = desired_class, check_size = geco_check_size)

            ori_pop_size = len(population)

            crossover_inplace(population, checked_set, components, generation, pop_size = max_num_population)
            mutation_inplace(population, checked_set, components, generation, pop_size = min(ori_pop_size, max_generation))

            selection_inplace(population, data, orig_instance, space, classifier, checked_set, pop_size = max_num_population, epslin = epslin, desired_class = desired_class, sample_num = sample_num)

            converge = check_converge(population, generation, geco_mutation)
            if geco_mutation and converge:
                if population[0].geco_verified:
                    population[0].geco_verified = False
                    changed = mutation_geco_inplace(population, checked_set, data, orig_instance, classifier, plaf_program, fidx_gidx, components, generation, desired_class = desired_class, check_size = 1)

                    selection_inplace(population, data, orig_instance, space, classifier, checked_set, pop_size = max_num_population, epslin = epslin, desired_class = desired_class, sample_num = sample_num)
                    if changed:
                        converge = False
                else:
                    converge = False

        if geco_mutation and reduction:
            geco_reduction_inplace(population, plaf_program, orig_instance, data, classifier, desired_class)

        return population, generation
    else:
        prep_time = 0.0
        selection_time = 0.0
        mutation_time = 0.0
        crossover_time = 0.0
        geco_init_time = 0.0
        geco_mutation_time = 0.0
        num_explored = 0
        num_explored_geco = 0
        reduction_time = 0.0

        start_time = time.time()
        groups, fidx_gidx = initRuleFeatureGroups(plaf_program, data)
        components = initailComponents(groups)
        prep_time += (time.time() - start_time)

        population, checked_set = [], set()

        start_time = time.time()
        population, population_bit_sets = initialPopulation_naive(components)
        checked_set = {tuple(b) for b in population_bit_sets}
        prep_time += (time.time() - start_time)

        if geco_initial:
            start_time = time.time()
            population, checked_set = initialPopulationWithGeco(data, orig_instance, plaf_program, classifier, fidx_gidx, components, population, checked_set, desired_class = desired_class)
            geco_init_time = (time.time() - start_time)
            num_explored_geco += 1

        start_time = time.time()
        space = initialSpace(orig_instance, data)
        prep_time += (time.time() - start_time)

        num_explored += max(0, len(population) - max_num_population)
        start_time = time.time()
        selection_inplace(population, data, orig_instance, space, classifier, checked_set, pop_size = max_num_population, epslin = epslin, desired_class = desired_class, sample_num = sample_num)
        selection_time += (time.time() - start_time)

        generation = 0
        converge = False

        geco_checked = True
        if epslin == 0.0 and geco_mutation:
            geco_checked = False

        last_geco = 0

        while generation < max_generation and (not converge or len(population) < 1 or population[0].precision < 1 - epslin):
            generation += 1

            if geco_mutation and generation - last_geco > 3:
                last_geco = generation
                num_explored_geco += min(len(population), geco_check_size)
                start_time = time.time()
                mutation_geco_inplace(population, checked_set, data, orig_instance, classifier, plaf_program, fidx_gidx, components, generation, desired_class = desired_class, check_size = geco_check_size)
                geco_mutation_time += (time.time() - start_time)

            ori_pop_size = len(population)

            start_time = time.time()
            crossover_inplace(population, checked_set, components, generation, pop_size = max_num_population)
            crossover_time += (time.time() - start_time)

            start_time = time.time()
            mutation_inplace(population, checked_set, components, generation, pop_size = min(ori_pop_size, max_generation))
            mutation_time += (time.time() - start_time)

            num_explored += max(0, len(population) - max_num_population)
            start_time = time.time()
            selection_inplace(population, data, orig_instance, space, classifier, checked_set, pop_size = max_num_population, epslin = epslin, desired_class = desired_class, sample_num = sample_num)
            selection_time += (time.time() - start_time)

            converge = check_converge(population, generation, geco_mutation)
            if geco_mutation and converge:
                if population[0].geco_verified:
                    population[0].geco_verified = False
                    start_time = time.time()
                    changed = mutation_geco_inplace(population, checked_set, data, orig_instance, classifier, plaf_program, fidx_gidx, components, generation, desired_class = desired_class, check_size = 1)
                    geco_mutation_time += (time.time() - start_time)

                    num_explored += max(0, len(population) - max_num_population)
                    start_time = time.time()
                    selection_inplace(population, data, orig_instance, space, classifier, checked_set, pop_size = max_num_population, epslin = epslin, desired_class = desired_class, sample_num = sample_num)
                    selection_time += (time.time() - start_time)
                    if changed:
                        converge = False
                else:
                    converge = False

        if geco_mutation and reduction:
            start_time = time.time()
            geco_reduction_inplace(population, plaf_program, orig_instance, data, classifier, desired_class)
            reduction_time = (time.time() - start_time)

        return population, generation, num_explored + num_explored_geco, num_explored_geco, prep_time, selection_time, mutation_time, crossover_time, geco_init_time, geco_mutation_time, reduction_time

def initialPopulation_naive(components: List[RuleComponent]):
    initial_pop: List[Rule] = []
    population_bit_sets: List[np.ndarray] = []

    for index, component in enumerate(components):
        rule = initialRule(len(components))
        rule.rule_components.append(component)
        rule.bit_representation[index] = True
        if component.group.categorical:
            if component.direction:
                rule.rule_components.append(components[index + 1])
                rule.bit_representation[index + 1] = True
            else:
                continue
        initial_pop.append(rule)
        population_bit_sets.append(rule.bit_representation)

    return initial_pop, population_bit_sets

def initialSpace(orig_instance: pd.Series, data: pd.DataFrame):
    space = []

    for feature in data.columns:
        feature_space = []
        all_values = data[feature].values
        # Julia: all_values[all_values .>= orig_instance[feature]]
        feature_space.append(all_values[all_values >= orig_instance[feature]])
        feature_space.append(all_values[all_values <= orig_instance[feature]])
        space.append(feature_space)

    return space

def sort_rule(rule: Rule):
    return rule.score


def selection_inplace(population: List[Rule], data: pd.DataFrame, orig_instance: pd.Series, space, classifier, checked_set: Set[Tuple[bool, ...]], pop_size: int = 10, epslin = 0.95, desired_class = 1, sample_num = 1000):
    for i in range(len(population)):
        if population[i].score != 0:
            continue

        evaluation(data, population[i], orig_instance, space, classifier, epslin = epslin, desired_class = desired_class, sample_num = sample_num)

    population.sort(key=sort_rule, reverse=True)

    if len(population) > pop_size:
        # Keep pop_size elements, but cleanup the checked_set for removed elements
        for i in range(pop_size, len(population)):
            if population[i].score < 0.5:
                break
            else:
                bit_tuple = tuple(population[i].bit_representation)
                if bit_tuple in checked_set:
                    checked_set.remove(bit_tuple)

        del population[pop_size:]

def mutation_inplace(population: List[Rule], checked_set: Set[Tuple[bool, ...]], components: List[RuleComponent], generation: int, max_num_samples: int = 3, pop_size = 20):
    row_num = min(len(population), pop_size)
    col_num = len(components)

    for row_index in range(row_num):
        rule: Rule = population[row_index]

        possible_components = []
        for i in range(col_num):
            if not rule.bit_representation[i]:
                possible_components.append(i)

        if not possible_components:
            continue

        num_to_sample = min(len(possible_components), max_num_samples)
        changes = random.sample(possible_components, num_to_sample)

        for index in changes:
            mutated_rule: Rule = copy.deepcopy(rule)
            mutated_rule.generation = generation
            mutated_rule.geco_verified = False
            mutated_rule.geco_generated = False
            mutated_rule.score = 0

            # check whether the group is categorical
            if components[index].group.categorical:
                # index/2 equivalent
                group_index = int(np.ceil((index + 1) / 2)) - 1 # 0-indexed adjustment

                # Handling 0-indexed group logic for categorical pairs
                idx_1 = group_index * 2
                idx_2 = group_index * 2 + 1

                if not mutated_rule.bit_representation[idx_1]:
                    mutated_rule.bit_representation[idx_1] = True
                    mutated_rule.rule_components.append(components[idx_1])
                if not mutated_rule.bit_representation[idx_2]:
                    mutated_rule.bit_representation[idx_2] = True
                    mutated_rule.rule_components.append(components[idx_2])
            else:
                mutated_rule.bit_representation[index] = True
                mutated_rule.rule_components.append(components[index])

            bit_tuple = tuple(mutated_rule.bit_representation)
            if bit_tuple not in checked_set:
                mutated_rule.rule_components.sort(key=sort_components)
                population.append(mutated_rule)
                checked_set.add(bit_tuple)

def crossover_inplace(population: List[Rule], checked_set: Set[Tuple[bool, ...]], components: List[RuleComponent], generation: int, pop_size = 20):
    row_num = min(len(population), pop_size)
    col_num = len(components)
    for parent1_index in range(row_num):
        for parent2_index in range(parent1_index + 1, row_num):

            mutated_1: Rule = copy.deepcopy(population[parent1_index])
            mutated_1.score = 0
            mutated_1.generation = generation
            mutated_1.geco_verified = False
            mutated_1.geco_generated = False

            mutated_2: Rule = copy.deepcopy(population[parent2_index])
            mutated_2.generation = generation
            mutated_2.geco_verified = False
            mutated_2.geco_generated = False
            mutated_2.score = 0

            choice_1 = []
            choice_2 = []

            for i in range(col_num):
                if mutated_1.bit_representation[i] and not mutated_2.bit_representation[i]:
                    choice_2.append(i)
                elif mutated_2.bit_representation[i] and not mutated_1.bit_representation[i]:
                    choice_1.append(i)

            if choice_1:
                add_component = random.choice(choice_1)
                if components[add_component].group.categorical:
                    group_index = int(np.ceil((add_component + 1) / 2)) - 1
                    idx_1, idx_2 = group_index * 2, group_index * 2 + 1
                    mutated_1.bit_representation[idx_1] = True
                    mutated_1.rule_components.append(components[idx_1])
                    mutated_1.bit_representation[idx_2] = True
                    mutated_1.rule_components.append(components[idx_2])
                else:
                    mutated_1.bit_representation[add_component] = True
                    mutated_1.rule_components.append(components[add_component])

                bit_tuple_1 = tuple(mutated_1.bit_representation)
                if bit_tuple_1 not in checked_set:
                    mutated_1.rule_components.sort(key=sort_components)
                    population.append(mutated_1)
                    checked_set.add(bit_tuple_1)

            if choice_2:
                add_component = random.choice(choice_2)
                if components[add_component].group.categorical:
                    group_index = int(np.ceil((add_component + 1) / 2)) - 1
                    idx_1, idx_2 = group_index * 2, group_index * 2 + 1
                    mutated_2.bit_representation[idx_1] = True
                    mutated_2.rule_components.append(components[idx_1])
                    mutated_2.bit_representation[idx_2] = True
                    mutated_2.rule_components.append(components[idx_2])
                else:
                    mutated_2.bit_representation[add_component] = True
                    mutated_2.rule_components.append(components[add_component])

                bit_tuple_2 = tuple(mutated_2.bit_representation)
                if bit_tuple_2 not in checked_set:
                    mutated_2.rule_components.sort(key=sort_components)
                    population.append(mutated_2)
                    checked_set.add(bit_tuple_2)


def evaluation(data: pd.DataFrame, rule: Rule, orig_instance: pd.Series, space, classifier, epslin = 0.05, desired_class = 1, sample_num = 1000):
    samples = sample_data(data, rule, orig_instance, space, number = sample_num)

    preds: np.ndarray = GeCo.score(classifier, samples, desired_class, extra_col = 0)

    # count(i->(i<0.5), preds)
    precision = np.sum(preds < 0.5) / len(preds)

    rule.precision = precision
    bit_count = np.sum(rule.bit_representation)
    bit_len = len(rule.bit_representation)

    if rule.geco_verified:
        rule.score = 1.0 / bit_len * 0.25 + 0.5 - bit_count / bit_len * 0.5 + precision * 0.5
    elif precision < 1.0 - epslin:
        if precision < 0.25 and rule.geco_generated:
            precision += bit_count / bit_len
        rule.score = precision * 0.5
    else:
        rule.score = 0.5 - bit_count / bit_len * 0.5 + precision * 0.5

def sample_data(data: pd.DataFrame, rule: Rule, orig_instance: pd.Series, space, number: int = 1000):
    rows = np.random.randint(0, len(data), number)
    sample_df = data.iloc[rows].copy()

    for idx, component in enumerate(rule.rule_components):
        direction = 0 # 0 means >=, 1 means <=, 2 means =
        if idx != 0 and (component.group_index == rule.rule_components[idx - 1].group_index):
            continue

        if idx != len(rule.rule_components) - 1 and component.group_index == rule.rule_components[idx + 1].group_index:
            # exact same
            direction = 2
        elif not component.direction:
            direction = 1

        for fidx, feature in zip(component.group.indexes, component.group.features):
            orig_val = orig_instance[feature]
            if direction == 2: # ==
                sample_df[feature] = orig_val
            elif direction == 1: # <=
                violates_mask = sample_df[feature] > orig_val
                violates_num = np.sum(violates_mask)
                if violates_num > 0:
                    replacements = np.random.choice(space[fidx][1], violates_num)
                    sample_df.loc[violates_mask, feature] = replacements
            else: # >=
                violates_mask = sample_df[feature] < orig_val
                violates_num = np.sum(violates_mask)
                if violates_num > 0:
                    replacements = np.random.choice(space[fidx][0], violates_num)
                    sample_df.loc[violates_mask, feature] = replacements

    return sample_df


def initialPopulationWithGeco(data: pd.DataFrame, orig_instance: pd.Series, plaf_program: PLAFProgram, classifier, fidx_gidx, components, population: List[Rule], checked_set: Set[Tuple[bool, ...]], desired_class = 1, num_actions = 10):
    explains, _ = GeCo.explain(orig_instance, data, plaf_program, classifier, desired_class = desired_class)

    # helper functions for GeCo results (assumed behavior based on names)
    def fetch_feature_actions(explains, orig_instance):
        return [] # Placeholder

    def actions_reduced_groups(feature_actions, fidx_gidx):
        return [] # Placeholder

    feature_actions = fetch_feature_actions(explains, orig_instance)
    group_actions = actions_reduced_groups(feature_actions, fidx_gidx)

    action_components: List[ActionComponent] = []

    backtrace_geco(group_actions, 0, action_components, components, population, checked_set, 0)
    return population, checked_set

def action_to_rule(action_components: List[ActionComponent], components: List[RuleComponent], component_num: int):
    rule = Rule(component_num)
    for ac in action_components:
        # Map back to rule components based on index and direction
        # ac.index is group index, ac.direction is 1 or 0
        comp_idx = ac.index * 2 if ac.direction == 1 else ac.index * 2 + 1
        rule.bit_representation[comp_idx] = True
        rule.rule_components.append(components[comp_idx])
    rule.rule_components.sort(key=sort_components)
    return rule

def backtrace_geco(actions: List[Action], index: int, action_components: List[ActionComponent], components: List[RuleComponent], population: List[Rule], checked_set: Set[Tuple[bool, ...]], generation: int, max_action = 5):
    # Completion of the truncated Julia function
    if index >= min(len(actions), max_action):
        if action_components:
            rule = action_to_rule(action_components, components, len(components))
            rule.generation = generation
            rule.geco_generated = True
            bit_tuple = tuple(rule.bit_representation)
            if bit_tuple not in checked_set:
                population.append(rule)
                checked_set.add(bit_tuple)
        return

    # Option 1: Don't include this action
    backtrace_geco(actions, index + 1, action_components, components, population, checked_set, generation, max_action)

    # Option 2: Include components from this action
    for comp in actions[index].components:
        action_components.append(comp)
        backtrace_geco(actions, index + 1, action_components, components, population, checked_set, generation, max_action)
        action_components.pop()

# Dummy implementations for mutation_geco and geco_reduction to make generate_rules complete
def mutation_geco_inplace(*args, **kwargs):
    return False

def geco_reduction_inplace(*args, **kwargs):
    pass


#### GeCo.jl

In [ ]:
import pandas as pd
import numpy as np
import time
import json
from typing import List, Union, Tuple, Any

# Required internal imports based on the original Julia 'include' statements
# These assume a Python package structure where these files exist as modules.
from .components.plaf import PLAFProgram, initPLAF
from .data_manager.DataManager import DataManager, materialize
from .components.feasible_space import (
    feasibleSpace,
    initDomains,
    ground,
    FeatureGroup,
    applyConstraint,
    initGroups
)
from .components.action_cascade import GroundedImplication, actionCascade
from .classifier.random_forest_eval import (
    initRandomForestEval,
    initPartialRandomForestEval,
    predict as rf_predict
)
from .classifier.mlp_eval import MLPEvaluation, initMLPEval, predict as mlp_predict
from .classifier.fico import predict as fico_predict, FICO_CLASSIFIER
from .components.score import score
from .components.distance import (
    distance,
    distanceFeatureGroup,
    minimumObservableCounterfactual,
    observableCounterfactuals
)
from .components.initial_population import initialPopulation
from .components.crossover import crossover
from .components.mutation import mutation
from .components.selection import selection

# Constants
DEFAULT_NORM_RATIO = [0.25, 0.25, 0.25, 0.25]
NUM_EXTRA_COL = 4
EXTRA_COLS = ['score', 'outc', 'estcf', 'mod']
NUM_EXTRA_FEASIBLE_SPACE_COL = 2

def explain(orig_instance: pd.Series, data: pd.DataFrame, program: PLAFProgram, classifier,
            desired_class=1,
            k: int = 100,
            max_num_generations: int = 100,
            min_num_generations: int = 3,
            max_num_samples: int = 5,
            max_samples_init: int = 20,
            convergence_k: int = 5,
            norm_ratio: List[float] = DEFAULT_NORM_RATIO,
            domains: List[pd.DataFrame] = None,
            compress_data: bool = False,
            return_df: bool = False,
            ablation: bool = False,
            run_crossover: bool = True,
            run_mutation: bool = True,
            size_distance_temp: int = 100_000,
            verbose: bool = False):

    if domains is None:
        domains = []

    distance_temp = np.empty(size_distance_temp, dtype=np.float64)
    representation_size = np.zeros(max_num_generations + 1, dtype=np.int64)

    generation: int = 0
    count: int = 0
    converged: bool = False

    if not ablation and verbose:
        # Compute the feasible space for each feature group
        print("-- Time feasible space:\t", end="")
        start_time = time.perf_counter()
        feasible_space = feasibleSpace(data, orig_instance, program, domains=domains)
        print(f"{time.perf_counter() - start_time:.6f} seconds")

        print("-- Time init pop:\t", end="")
        start_time = time.perf_counter()
        population = initialPopulation(orig_instance, feasible_space, compress_data=compress_data, max_num_samples=max_samples_init)
        print(f"{time.perf_counter() - start_time:.6f} seconds")

        if compress_data:
            count += population.shape[0]
            representation_size[0] = population.shape[1]
        else:
            count += len(population)
            representation_size[0] = population.shape[0] * population.shape[1]

        print("-- Time selection:\t", end="")
        start_time = time.perf_counter()
        selection(population, k, orig_instance, feasible_space, classifier, desired_class,
                  norm_ratio=norm_ratio,
                  distance_temp=distance_temp)
        print(f"{time.perf_counter() - start_time:.6f} seconds")

        start_loop_time = time.perf_counter()
        while generation < min_num_generations or (not converged and generation < max_num_generations):
            print(f"Generation: {generation + 1}")

            print("-- Time Crossover:\t", end="")
            start_time = time.perf_counter()
            crossover(population, orig_instance, feasible_space)
            print(f"{time.perf_counter() - start_time:.6f} seconds")

            print("-- Time Mutation:\t", end="")
            start_time = time.perf_counter()
            mutation(population, feasible_space, max_num_samples=max_num_samples)
            print(f"{time.perf_counter() - start_time:.6f} seconds")

            if compress_data:
                count += max(0, population.shape[0] - k)
                representation_size[generation + 1] = population.shape[1]
            else:
                count += max(0, len(population) - k)
                representation_size[generation + 1] = population.shape[0] * population.shape[1]

            print("-- Time Selection:\t", end="")
            start_time = time.perf_counter()
            converged = selection(population, k, orig_instance, feasible_space, classifier, desired_class,
                                  norm_ratio=norm_ratio, distance_temp=distance_temp, convergence_k=convergence_k)
            print(f"{time.perf_counter() - start_time:.6f} seconds")

            generation += 1

        print(f"Number of generated counterfactuals: {count}\n" +
              f"Number of generations:               {generation}")

    elif not ablation:
        feasible_space = feasibleSpace(data, orig_instance, program, domains=domains)
        population = initialPopulation(orig_instance, feasible_space, compress_data=compress_data)

        if compress_data:
            count += population.shape[0]
            representation_size[0] = population.shape[1]
        else:
            count += len(population)
            representation_size[0] = population.shape[0] * population.shape[1]

        selection(population, k, orig_instance, feasible_space, classifier, desired_class,
                  norm_ratio=norm_ratio,
                  distance_temp=distance_temp)

        while generation < min_num_generations or (not converged and generation < max_num_generations):
            crossover(population, orig_instance, feasible_space)
            mutation(population, feasible_space, max_num_samples=max_num_samples)

            pop_shape = population.shape
            count += max(0, pop_shape[0] - k)
            if compress_data:
                representation_size[generation + 1] = pop_shape[1]
            else:
                representation_size[generation + 1] = pop_shape[0] * pop_shape[1]

            converged = selection(population, k, orig_instance, feasible_space, classifier, desired_class,
                                  norm_ratio=norm_ratio, distance_temp=distance_temp, convergence_k=convergence_k)

            generation += 1
    else:
        selection_time = 0.0
        mutation_time = 0.0
        crossover_time = 0.0

        feasible_space = feasibleSpace(data, orig_instance, program, domains=domains)

        start_prep = time.perf_counter()
        population = initialPopulation(orig_instance, feasible_space, compress_data=compress_data)

        if compress_data:
            count += population.shape[0]
            representation_size[0] = population.shape[1]
        else:
            count += len(population)
            representation_size[0] = population.shape[0] * population.shape[1]

        start_sel = time.perf_counter()
        selection(population, k, orig_instance, feasible_space, classifier, desired_class,
                  norm_ratio=norm_ratio,
                  distance_temp=distance_temp)
        stime = time.perf_counter() - start_sel

        prep_time = (time.perf_counter() - start_prep) # Including the selection time as per Julia logic

        while generation < min_num_generations or (not converged and generation < max_num_generations):
            if run_crossover:
                start_c = time.perf_counter()
                crossover(population, orig_instance, feasible_space)
                crossover_time += (time.perf_counter() - start_c)

            if run_mutation:
                start_m = time.perf_counter()
                mutation(population, feasible_space, max_num_samples=max_num_samples)
                mutation_time += (time.perf_counter() - start_m)

            pop_shape = population.shape
            count += max(0, pop_shape[0] - k)
            if compress_data:
                representation_size[generation + 1] = pop_shape[1]
            else:
                representation_size[generation + 1] = pop_shape[0] * pop_shape[1]

            start_s = time.perf_counter()
            converged = selection(population, k, orig_instance, feasible_space, classifier, desired_class,
                                  norm_ratio=norm_ratio,
                                  distance_temp=distance_temp,
                                  convergence_k=convergence_k)
            stime = time.perf_counter() - start_s
            selection_time += stime

            generation += 1

        if return_df and compress_data:
            population = materialize(population)
            population.sort_values(by='score', inplace=True)

        return population, count, generation, representation_size, prep_time, selection_time, mutation_time, crossover_time

    if return_df and compress_data:
        population = materialize(population)
        population.sort_values(by='score', inplace=True)

    return population, count, generation, representation_size


def actions(counterfactuals: Union[pd.DataFrame, DataManager], orig_instance: pd.Series, num_actions: int = 5):
    if isinstance(counterfactuals, pd.DataFrame):
        for idx in range(min(num_actions, len(counterfactuals))):
            cf = counterfactuals.iloc[idx]
            print(f"\n------- COUNTERFACTUAL {idx + 1}\nDesired Outcome: {cf['outc']},\tScore: {cf['score']}")
            for feature in orig_instance.index:
                delta = cf[feature] - orig_instance[feature]
                if delta != 0:
                    print(f"{feature} : \t{orig_instance[feature]} => {cf[feature]}")
    elif isinstance(counterfactuals, DataManager):
        # Turn DataManager into a DataFrame
        df = materialize(counterfactuals)
        df.sort_values(by='score', inplace=True)

        for idx in range(min(num_actions, len(df))):
            cf = df.iloc[idx]
            print(f"\n------- COUNTERFACTUAL {idx + 1}\nDesired Outcome: {cf['outc']},\tScore: {cf['score']}")
            for feature in orig_instance.index:
                delta = cf[feature] - orig_instance[feature]
                if delta != 0:
                    print(f"{feature} : \t{orig_instance[feature]} => {cf[feature]}")


### Adult Data

#### compare_adult.jl

In [ ]:
import pandas as pd
import numpy as np
import time
import joblib
import math
import copy
from sklearn.base import ClassifierMixin

# ==============================================================================
# Helper methods
# ==============================================================================

class RuleFeatureGroup:
    def __init__(self, features, indexes, categorical):
        self.features = features
        self.indexes = indexes
        self.categorical = categorical

    def __eq__(self, other):
        if not isinstance(other, RuleFeatureGroup):
            return False
        return self.features == other.features and self.indexes == other.indexes

    def __hash__(self):
        return hash((tuple(self.features), tuple(self.indexes)))

class RuleComponent:
    def __init__(self, group, direction, index, group_index):
        self.group = group
        self.direction = direction  # True for >=, False for <=
        self.index = index
        self.group_index = group_index

class Rule:
    def __init__(self, component_num):
        self.rule_components = []
        self.bit_representation = np.zeros(component_num, dtype=bool)
        self.score = 0.0
        self.precision = 0.0
        self.generation = 0
        self.geco_verified = False
        self.geco_generated = False

class ActionComponent:
    def __init__(self, index, direction):
        self.index = index
        self.direction = direction # 1 -> smaller, 2 -> larger (matching Julia logic)

    def __eq__(self, other):
        if not isinstance(other, ActionComponent):
            return False
        return self.index == other.index and self.direction == other.direction

class Action:
    def __init__(self, components):
        self.components = components

class Constraint:
    def __init__(self, feature, op, value):
        self.feature = feature
        self.op = op
        self.value = value

    def copy(self):
        return Constraint(self.feature, self.op, self.value)

class PLAFProgram:
    def __init__(self, other=None):
        if other:
            self.groups = list(other.groups)
            self.constraints = [c.copy() for c in other.constraints]
        else:
            self.groups = []
            self.constraints = []

# ==============================================================================
# GeCo
# ==============================================================================

class GeCo:
    @staticmethod
    def score(classifier, X, desired_class, extra_col=0):
        # Predict probability of desired_class
        if hasattr(classifier, "predict_proba"):
            probs = classifier.predict_proba(X)
            # In scikit-learn, predict_proba returns [n_samples, n_classes]
            return probs[:, desired_class]
        else:
            preds = classifier.predict(X)
            return (preds == desired_class).astype(float)

    @staticmethod
    def explain(orig_instance, data, plaf, classifier, desired_class=1, **kwargs):
        # In this context, explain is used to find counterfactuals that satisfy
        # constraints 'plaf'.

        # We sample points from 'data' that satisfy 'plaf' constraints
        # then check if classifier predicts 'desired_class'.

        # 1. Filter data by constraints
        filtered_data = data.copy()
        for c in plaf.constraints:
            if c.op == "==":
                filtered_data = filtered_data[filtered_data[c.feature] == c.value]
            elif c.op == "<=":
                filtered_data = filtered_data[filtered_data[c.feature] <= c.value]
            elif c.op == ">=":
                filtered_data = filtered_data[filtered_data[c.feature] >= c.value]

        if len(filtered_data) == 0:
            return pd.DataFrame(columns=data.columns.tolist() + ['outc']),

        # 2. Score these points
        # For simplicity, we sample up to 1000 points if the filtered data is large
        if len(filtered_data) > 1000:
            sample_subset = filtered_data.sample(1000)
        else:
            sample_subset = filtered_data

        scores = GeCo.score(classifier, sample_subset, desired_class)

        # outc is True if score >= 0.5 (meaning it is the desired class)
        sample_subset = sample_subset.copy()
        sample_subset['outc'] = (scores >= 0.5)

        # Return only found counterfactuals
        counterfactuals = sample_subset[sample_subset['outc'] == True]

        return counterfactuals,

def explain(orig_instance, data, plaf, classifier, desired_class=1, **kwargs):
    return GeCo.explain(orig_instance, data, plaf, classifier, desired_class, **kwargs)

# ==============================================================================
# GeneticCF.jl implementation details
# ==============================================================================

def initRuleFeatureGroups(prog, data):
    groups = []
    fidx_groupidx = np.zeros(data.shape[1], dtype=np.int32) - 1
    includedFeatures = set()
    col_names = list(data.columns)

    for group in prog.groups:
        indexes = []
        for feature in group:
            if feature not in col_names:
                raise ValueError(f"The feature {feature} does not occur in the input data.")
            if feature in includedFeatures:
                raise ValueError("Each feature can be in at most one group!")
            includedFeatures.add(feature)
            indexes.append(col_names.index(feature))

        groups.append(RuleFeatureGroup(list(group), indexes, True))

    for feature in col_names:
        if feature not in includedFeatures:
            idx = col_names.index(feature)
            # Check if categorical/multiclass (simplified)
            is_categorical = data[feature].dtype == 'object' or str(data[feature].dtype) == 'category'
            groups.append(RuleFeatureGroup([feature], [idx], is_categorical))

    for gidx, group in enumerate(groups):
        for fidx in group.indexes:
            fidx_groupidx[fidx] = gidx

    return groups, fidx_groupidx

def initailComponents(groups):
    components = []
    for idx, group in enumerate(groups):
        components.append(RuleComponent(group, True, 2 * idx, idx))
        components.append(RuleComponent(group, False, 2 * idx + 1, idx))
    return components

def initialRule(component_num):
    return Rule(component_num)

def print_rule(orig_instance, rule):
    print(f"The precision of the rule is: {rule.precision}")
    print(f"This rule has {len(rule.rule_components)} components")

    included_groups = set()
    index = 1

    for rule_component in rule.rule_components:
        if rule_component.group in included_groups:
            continue

        bidirection = False
        for other in rule.rule_components:
            if other.group_index == rule_component.group_index and other.direction != rule_component.direction:
                bidirection = True
                break

        if bidirection:
            if rule_component.direction:
                continue
            print(f"\t Component {index} and {index + 1} defines exact match on the feature group:")
            index += 1
        elif rule_component.direction: # >=
            print(f"\t Component {index} defines lower bound on the feature group:")
        else:
            print(f"\t Component {index} defines upper bound on the feature group:")

        for feature in rule_component.group.features:
            val = orig_instance[feature]
            if bidirection:
                print(f"\t\t For feature {feature}: x = {val}")
            elif rule_component.direction:
                print(f"\t\t For feature {feature}: x >= {val}")
            else:
                print(f"\t\t For feature {feature}: x <= {val}")

        index += 1
        included_groups.add(rule_component.group)

def sort_action_cardinality(action):
    return len(action.components)

def check_converge(population, generation, geco_mutation):
    if not population:
        return False
    if geco_mutation and population[0].geco_verified and population[0].generation + 3 <= generation:
        return True

    i = 0
    for rule in population:
        if rule.generation + 1 >= generation:
            return False
        if i >= 2 and population[0].geco_verified: # Index 2 means 3rd element
            return True
        if i >= 4:
            return True
        i += 1
    return True

def initialSpace(orig_instance, data):
    space = []
    for feature in data.columns:
        all_values = data[feature].values
        greater_eq = all_values[all_values >= orig_instance[feature]]
        less_eq = all_values[all_values <= orig_instance[feature]]
        space.append([greater_eq, less_eq])
    return space

def evaluation(data, rule, orig_instance, space, classifier, epslin=0.05, desired_class=1, sample_num=1000):
    samples = sample_data(data, rule, orig_instance, space, number=sample_num)
    preds = GeCo.score(classifier, samples, desired_class)

    precision = np.sum(preds < 0.5) / len(preds)
    rule.precision = precision

    comp_len = len(rule.bit_representation)
    rule_sum = np.sum(rule.bit_representation)

    if rule.geco_verified:
        rule.score = (1.0 / comp_len) * 0.25 + 0.5 - (rule_sum / comp_len) * 0.5 + precision * 0.5
    elif precision < 1.0 - epslin:
        if precision < 0.25 and rule.geco_generated:
            precision += (rule_sum / comp_len)
        rule.score = precision * 0.5
    else:
        rule.score = 0.5 - (rule_sum / comp_len) * 0.5 + precision * 0.5

def sample_data(data, rule, orig_instance, space, number=1000):
    rows = np.random.randint(0, len(data), number)
    sample_df = data.iloc[rows].copy()

    # Sort components to detect bi-directional (exact match) constraints
    sorted_comps = sorted(rule.rule_components, key=lambda x: (x.group_index, not x.direction))

    i = 0
    while i < len(sorted_comps):
        comp = sorted_comps[i]
        direction_mode = 0 # 0: >=, 1: <=, 2: ==

        if i + 1 < len(sorted_comps) and comp.group_index == sorted_comps[i+1].group_index:
            direction_mode = 2
            i += 1
        elif not comp.direction:
            direction_mode = 1

        group = comp.group
        for fidx, feature in zip(group.indexes, group.features):
            orig_val = orig_instance[feature]
            if direction_mode == 2:
                sample_df[feature] = orig_val
            elif direction_mode == 1: # <=
                mask = sample_df[feature] > orig_val
                violates_count = mask.sum()
                if violates_count > 0:
                    replaces = np.random.choice(space[fidx][1], violates_count)
                    sample_df.loc[mask, feature] = replaces
            else: # >=
                mask = sample_df[feature] < orig_val
                violates_count = mask.sum()
                if violates_count > 0:
                    replaces = np.random.choice(space[fidx][0], violates_count)
                    sample_df.loc[mask, feature] = replaces
        i += 1
    return sample_df

def selection_fn(population, data, orig_instance, space, classifier, checked_set, pop_size=10, epslin=0.95, desired_class=1, sample_num=1000):
    for rule in population:
        if rule.score != 0:
            continue
        evaluation(data, rule, orig_instance, space, classifier, epslin=epslin, desired_class=desired_class, sample_num=sample_num)

    population.sort(key=lambda x: x.score, reverse=True)

    if len(population) > pop_size:
        for i in range(pop_size, len(population)):
            if population[i].score < 0.5:
                break
            else:
                checked_set.discard(tuple(population[i].bit_representation))
        del population[pop_size:]

def mutation_fn(population, checked_set, components, generation, max_num_samples=3, pop_size=20):
    row_num = min(len(population), pop_size)
    col_num = len(components)

    for row_index in range(row_num):
        rule = population[row_index]
        possible_components = [i for i in range(col_num) if not rule.bit_representation[i]]

        if not possible_components: continue
        changes = np.random.choice(possible_components, min(len(possible_components), max_num_samples), replace=False)

        for index in changes:
            mutated_rule = copy.deepcopy(rule)
            mutated_rule.generation = generation
            mutated_rule.geco_verified = False
            mutated_rule.geco_generated = False
            mutated_rule.score = 0.0

            if components[index].group.categorical:
                group_idx = index // 2
                mutated_rule.bit_representation[group_idx * 2] = True
                mutated_rule.bit_representation[group_idx * 2 + 1] = True
                # Re-build components list or just add
                found_0 = any(c.index == group_idx*2 for c in mutated_rule.rule_components)
                found_1 = any(c.index == group_idx*2 + 1 for c in mutated_rule.rule_components)
                if not found_0: mutated_rule.rule_components.append(components[group_idx * 2])
                if not found_1: mutated_rule.rule_components.append(components[group_idx * 2 + 1])
            else:
                mutated_rule.bit_representation[index] = True
                mutated_rule.rule_components.append(components[index])

            rep_tuple = tuple(mutated_rule.bit_representation)
            if rep_tuple not in checked_set:
                mutated_rule.rule_components.sort(key=lambda x: x.index)
                population.append(mutated_rule)
                checked_set.add(rep_tuple)

def crossover_fn(population, checked_set, components, generation, pop_size=20):
    row_num = min(len(population), pop_size)
    col_num = len(components)
    for p1_idx in range(row_num):
        for p2_idx in range(p1_idx + 1, row_num):
            for i in range(2): # 1 and 2 in Julia
                mut_idx = p1_idx if i == 0 else p2_idx
                other_idx = p2_idx if i == 0 else p1_idx

                mutated = copy.deepcopy(population[mut_idx])
                mutated.score = 0.0
                mutated.generation = generation
                mutated.geco_verified = False
                mutated.geco_generated = False

                other_rep = population[other_idx].bit_representation
                diff_indices = [j for j in range(col_num) if other_rep[j] and not mutated.bit_representation[j]]

                if diff_indices:
                    add_idx = np.random.choice(diff_indices)
                    if components[add_idx].group.categorical:
                        g_idx = add_idx // 2
                        mutated.bit_representation[g_idx * 2] = True
                        mutated.bit_representation[g_idx * 2 + 1] = True
                        # Add components if not present
                        for c_idx in [g_idx * 2, g_idx * 2 + 1]:
                            if not any(c.index == c_idx for c in mutated.rule_components):
                                mutated.rule_components.append(components[c_idx])
                    else:
                        mutated.bit_representation[add_idx] = True
                        mutated.rule_components.append(components[add_idx])

                    rep_tuple = tuple(mutated.bit_representation)
                    if rep_tuple not in checked_set:
                        mutated.rule_components.sort(key=lambda x: x.index)
                        population.append(mutated)
                        checked_set.add(rep_tuple)

def action_to_rule(action_components, components, generation):
    rule = Rule(len(components))
    rule.generation = generation
    for ac in action_components:
        c_idx = ac.index * 2
        if components[c_idx].group.categorical:
            if rule.bit_representation[c_idx]:
                continue
            rule.bit_representation[c_idx] = True
            rule.bit_representation[c_idx + 1] = True
            rule.rule_components.append(components[c_idx])
            rule.rule_components.append(components[c_idx + 1])
        else:
            actual_idx = c_idx if ac.direction == 2 else c_idx + 1
            if rule.bit_representation[actual_idx]:
                continue
            rule.bit_representation[actual_idx] = True
            rule.rule_components.append(components[actual_idx])
    rule.geco_generated = True
    return rule

def backtrace_geco(actions, index, action_components, components, population, checked_set, generation, max_action=5):
    if index >= min(len(actions), max_action):
        rule = action_to_rule(action_components, components, generation)
        rep_tuple = tuple(rule.bit_representation)
        if rep_tuple not in checked_set:
            rule.rule_components.sort(key=lambda x: x.index)
            population.append(rule)
            checked_set.add(rep_tuple)
        return

    for ac in actions[index].components:
        action_components.append(ac)
        backtrace_geco(actions, index + 1, action_components, components, population, checked_set, generation, max_action)
        action_components.pop()

def fetch_feature_actions(counterfactuals, orig_instance):
    res = []
    for idx in range(len(counterfactuals)):
        cf = counterfactuals.iloc[idx]
        if not cf['outc']: continue
        action = Action([])
        for f_idx, feature in enumerate(orig_instance.index):
            delta = cf[feature] - orig_instance[feature]
            if delta != 0:
                direction = 2 if delta > 0 else 1
                action.components.append(ActionComponent(f_idx, direction))
        res.append(action)
    return res

def actions_reduced_groups(feature_actions, fidx_gidx):
    group_actions = []
    for fa in feature_actions:
        action = Action([])
        g_indices = set()
        for ac in fa.components:
            g_idx = fidx_gidx[ac.index]
            if g_idx in g_indices: continue
            g_indices.add(g_idx)
            action.components.append(ActionComponent(g_idx, ac.direction))
        group_actions.append(action)

    group_actions.sort(key=sort_action_cardinality)

    a_idx = 1
    while a_idx < len(group_actions):
        cur_action = group_actions[a_idx]
        covered = False
        for prev_idx in range(a_idx):
            prev_action = group_actions[prev_idx]
            # Check if prev_action is a subset of cur_action
            all_in = True
            for pac in prev_action.components:
                if pac not in cur_action.components:
                    all_in = False
                    break
            if all_in:
                covered = True
                break
        if covered:
            del group_actions[a_idx]
        else:
            a_idx += 1
    return group_actions

def rule_to_constrain(rule, plaf_program, orig_instance):
    plaf = PLAFProgram(plaf_program)
    current_components = []
    sorted_comps = sorted(rule.rule_components, key=lambda x: x.index)

    i = 0
    while i < len(sorted_comps):
        rc = sorted_comps[i]
        g_idx = rc.group_index

        mode = 0 # 0: >=, 1: <=, 2: ==
        if i + 1 < len(sorted_comps) and sorted_comps[i+1].group_index == g_idx:
            mode = 2
            current_components.append(ActionComponent(g_idx, 1))
            current_components.append(ActionComponent(g_idx, 2))
            i += 1
        elif rc.direction:
            mode = 0
            current_components.append(ActionComponent(g_idx, 2)) # Dir 2 is >= in Julia mapping
        else:
            mode = 1
            current_components.append(ActionComponent(g_idx, 1))

        for feature in rc.group.features:
            val = orig_instance[feature]
            op = "==" if mode == 2 else ("<=" if mode == 1 else ">=")
            # Note: parse(Float64) in Julia is just float() in Python
            plaf.constraints.append(Constraint(feature, op, float(val)))
        i += 1
    return plaf, current_components

def initialPopulation_naive(components):
    initial_pop = []
    population_set = set()
    for index, component in enumerate(components):
        rule = initialRule(len(components))
        rule.rule_components.append(component)
        rule.bit_representation[index] = True
        if component.group.categorical:
            if component.direction: # True direction component
                # Add the dual component (False direction)
                dual_comp = components[index + 1]
                rule.rule_components.append(dual_comp)
                rule.bit_representation[index + 1] = True
            else:
                continue
        initial_pop.append(rule)
        population_set.add(tuple(rule.bit_representation))
    return initial_pop, population_set

def initialPopulationWithGeco(data, orig_instance, plaf_program, classifier, fidx_gidx, components, population, checked_set, desired_class=1):
    explains_res = explain(orig_instance, data, plaf_program, classifier, desired_class=desired_class)
    counterfactuals = explains_res[0]
    feature_actions = fetch_feature_actions(counterfactuals, orig_instance)
    group_actions = actions_reduced_groups(feature_actions, fidx_gidx)
    backtrace_geco(group_actions, 0, [], components, population, checked_set, 0)
    return population, checked_set

def mutation_geco_fn(population, checked_set, data, orig_instance, classifier, plaf_program, fidx_gidx, components, generation, desired_class=1, check_size=10):
    row_num = min(len(population), check_size)
    idx = 0
    changed = False

    while row_num > 0:
        row_num -= 1
        if idx >= len(population): break
        rule = population[idx]
        if rule.geco_verified:
            idx += 1
            continue

        p, action_components = rule_to_constrain(rule, plaf_program, orig_instance)
        explains_res = explain(orig_instance, data, p, classifier, desired_class=desired_class)
        explains_df = explains_res[0]

        if len(explains_df) == 0 or not explains_df.iloc[0]['outc']:
            rule.geco_verified = True
            idx += 1
            continue

        rule.geco_verified = False
        cur_pop_size = len(population)
        changed = True

        f_actions = fetch_feature_actions(explains_df, orig_instance)
        g_actions = actions_reduced_groups(f_actions, fidx_gidx)
        backtrace_geco(g_actions, 0, action_components, components, population, checked_set, generation)

        if len(population) > cur_pop_size or cur_pop_size > 2:
            del population[idx]
        else:
            idx += 1

    return changed

def check_redundancy(rule, plaf_program, orig_instance, data, classifier, desired_class):
    if len(rule.rule_components) <= 1:
        return None

    for idx in range(len(rule.rule_components)):
        rule_new = copy.deepcopy(rule)
        deleted_comp = rule.rule_components[idx]
        # Remove component from rule_new
        rule_new.rule_components = [c for c in rule_new.rule_components if c.index != deleted_comp.index]
        rule_new.bit_representation[deleted_comp.index] = False

        if deleted_comp.group.categorical:
            # Categorical groups have two components;
            dual_idx = deleted_comp.index + (1 if deleted_comp.index % 2 == 0 else -1)
            rule_new.bit_representation[dual_idx] = False
            rule_new.rule_components = [c for c in rule_new.rule_components if c.index != dual_idx]

        p, action_components = rule_to_constrain(rule_new, plaf_program, orig_instance)
        explains_res = explain(orig_instance, data, p, classifier, desired_class=desired_class)
        explains_df = explains_res[0]

        if len(explains_df) == 0 or not explains_df.iloc[0]['outc']:
            rule_new.geco_verified = True
            rule_new.precision = 1.0
            return rule_new

    return None

def geco_reduction_fn(population, plaf_program, orig_instance, data, classifier, desired_class):
    rule_new = check_redundancy(population[0], plaf_program, orig_instance, data, classifier, desired_class)
    while rule_new is not None:
        population.insert(0, rule_new)
        rule_new = check_redundancy(population[0], plaf_program, orig_instance, data, classifier, desired_class)

def generate_rules(orig_instance, data, classifier, plaf_program,
                   epslin=0.0, max_num_population=50, geco_initial=False,
                   geco_mutation=False, desired_class=1, max_generation=50,
                   sample_num=1000, geco_check_size=5, reduction=False,
                   ablation=False):

    if not ablation:
        groups, fidx_gidx = initRuleFeatureGroups(plaf_program, data)
        components = initailComponents(groups)
        population, checked_set = initialPopulation_naive(components)

        if geco_initial:
            population, checked_set = initialPopulationWithGeco(data, orig_instance, plaf_program, classifier, fidx_gidx, components, population, checked_set, desired_class=desired_class)

        space = initialSpace(orig_instance, data)
        selection_fn(population, data, orig_instance, space, classifier, checked_set, pop_size=max_num_population, epslin=epslin, desired_class=desired_class, sample_num=sample_num)

        generation = 0
        converge = False
        last_geco = 0

        while generation < max_generation and (not converge or not population or population[0].precision < 1 - epslin):
            generation += 1

            if geco_mutation and (generation - last_geco > 3):
                last_geco = generation
                mutation_geco_fn(population, checked_set, data, orig_instance, classifier, plaf_program, fidx_gidx, components, generation, desired_class=desired_class, check_size=geco_check_size)

            ori_pop_size = len(population)
            crossover_fn(population, checked_set, components, generation, pop_size=max_num_population)
            mutation_fn(population, checked_set, components, generation, pop_size=min(ori_pop_size, max_generation))
            selection_fn(population, data, orig_instance, space, classifier, checked_set, pop_size=max_num_population, epslin=epslin, desired_class=desired_class, sample_num=sample_num)

            converge = check_converge(population, generation, geco_mutation)
            if geco_mutation and converge:
                if population[0].geco_verified:
                    population[0].geco_verified = False
                    changed = mutation_geco_fn(population, checked_set, data, orig_instance, classifier, plaf_program, fidx_gidx, components, generation, desired_class=desired_class, check_size=1)
                    selection_fn(population, data, orig_instance, space, classifier, checked_set, pop_size=max_num_population, epslin=epslin, desired_class=desired_class, sample_num=sample_num)
                    if changed: converge = False
                else:
                    converge = False

        if geco_mutation and reduction:
            geco_reduction_fn(population, plaf_program, orig_instance, data, classifier, desired_class)

        return population, generation
    else:
        # Implementing the timers as in Julia's ablation branch
        prep_time = 0.0
        selection_time = 0.0
        mutation_time = 0.0
        crossover_time = 0.0
        geco_init_time = 0.0
        geco_mutation_time = 0.0
        num_explored = 0
        num_explored_geco = 0
        reduction_time = 0.0

        t0 = time.perf_counter()
        groups, fidx_gidx = initRuleFeatureGroups(plaf_program, data)
        components = initailComponents(groups)
        prep_time += (time.perf_counter() - t0)

        t0 = time.perf_counter()
        population, checked_set = initialPopulation_naive(components)
        prep_time += (time.perf_counter() - t0)

        if geco_initial:
            t0 = time.perf_counter()
            population, checked_set = initialPopulationWithGeco(data, orig_instance, plaf_program, classifier, fidx_gidx, components, population, checked_set, desired_class=desired_class)
            geco_init_time = (time.perf_counter() - t0)
            num_explored_geco += 1

        t0 = time.perf_counter()
        space = initialSpace(orig_instance, data)
        prep_time += (time.perf_counter() - t0)

        num_explored += max(0, len(population) - max_num_population)
        t0 = time.perf_counter()
        selection_fn(population, data, orig_instance, space, classifier, checked_set, pop_size=max_num_population, epslin=epslin, desired_class=desired_class, sample_num=sample_num)
        selection_time += (time.perf_counter() - t0)

        generation = 0
        converge = False
        last_geco = 0

        while generation < max_generation and (not converge or not population or population[0].precision < 1 - epslin):
            generation += 1
            if geco_mutation and (generation - last_geco > 3):
                last_geco = generation
                num_explored_geco += min(len(population), geco_check_size)
                t0 = time.perf_counter()
                mutation_geco_fn(population, checked_set, data, orig_instance, classifier, plaf_program, fidx_gidx, components, generation, desired_class=desired_class, check_size=geco_check_size)
                geco_mutation_time += (time.perf_counter() - t0)

            ori_pop_size = len(population)
            t0 = time.perf_counter()
            crossover_fn(population, checked_set, components, generation, pop_size=max_num_population)
            crossover_time += (time.perf_counter() - t0)

            t0 = time.perf_counter()
            mutation_fn(population, checked_set, components, generation, pop_size=min(ori_pop_size, max_generation))
            mutation_time += (time.perf_counter() - t0)

            num_explored += max(0, len(population) - max_num_population)
            t0 = time.perf_counter()
            selection_fn(population, data, orig_instance, space, classifier, checked_set, pop_size=max_num_population, epslin=epslin, desired_class=desired_class, sample_num=sample_num)
            selection_time += (time.perf_counter() - t0)

            converge = check_converge(population, generation, geco_mutation)
            if geco_mutation and converge:
                if population[0].geco_verified:
                    population[0].geco_verified = False
                    t0 = time.perf_counter()
                    changed = mutation_geco_fn(population, checked_set, data, orig_instance, classifier, plaf_program, fidx_gidx, components, generation, desired_class=desired_class, check_size=1)
                    geco_mutation_time += (time.perf_counter() - t0)

                    num_explored += max(0, len(population) - max_num_population)
                    t0 = time.perf_counter()
                    selection_fn(population, data, orig_instance, space, classifier, checked_set, pop_size=max_num_population, epslin=epslin, desired_class=desired_class, sample_num=sample_num)
                    selection_time += (time.perf_counter() - t0)
                    if changed: converge = False
                else:
                    converge = False

        if geco_mutation and reduction:
            t0 = time.perf_counter()
            geco_reduction_fn(population, plaf_program, orig_instance, data, classifier, desired_class)
            reduction_time = (time.perf_counter() - t0)

        return population, generation, num_explored + num_explored_geco, num_explored_geco, prep_time, selection_time, mutation_time, crossover_time, geco_init_time, geco_mutation_time, reduction_time

def geco_for_rule(orig_instance, data, classifier, plaf_program, desired_class=1):
    groups, fidx_gidx = initRuleFeatureGroups(plaf_program, data)
    components = initailComponents(groups)

    initial_pop = []
    population_set = set()
    population, checked_set = initialPopulationWithGeco(data, orig_instance, plaf_program, classifier, fidx_gidx, components, initial_pop, population_set, desired_class=desired_class)

    population.sort(key=lambda x: len(x.rule_components), reverse=True)
    num_exp = 1

    while True:
        if not population:
            return None, -1
        rule = population.pop()
        num_exp += 1
        p_cur, action_components = rule_to_constrain(rule, plaf_program, orig_instance)
        explains_res = explain(orig_instance, data, p_cur, classifier, desired_class=desired_class)
        explains_df = explains_res[0]

        if len(explains_df) == 0 or not explains_df.iloc[0]['outc']:
            return rule, num_exp

        f_actions = fetch_feature_actions(explains_df, orig_instance)
        g_actions = actions_reduced_groups(f_actions, fidx_gidx)
        backtrace_geco(g_actions, 0, action_components, components, population, checked_set, 0)

        if len(population) == 0:
            return rule, -1

        population.sort(key=lambda x: len(x.rule_components), reverse=True)

# ==============================================================================
# Main script logic: geco_verify and run_exp
# ==============================================================================

def geco_verify(rules, orig_instance, plaf, X, classifier, desired_class):
    # verify using geco
    if not rules:
        return True
    rule = rules[0]

    # create the constraints for plaf by the rule
    p, _ = rule_to_constrain(rule, plaf, orig_instance)

    geco_explanation_res = explain(orig_instance, X, p, classifier, desired_class=desired_class)
    geco_explanation = geco_explanation_res[0]

    if len(geco_explanation) != 0 and geco_explanation.iloc[0]['outc']:
        return False
    return True

def run_exp(X, classifier):
    desired_class = 0

    plaf = PLAFProgram()
    # Equivalent to @GROUP(plaf, s for s in propertynames(X) if contains(string(s), "Relationship"))
    plaf.groups.append([s for s in X.columns if "Relationship" in str(s)])
    plaf.groups.append([s for s in X.columns if "Occupation" in str(s)])
    plaf.groups.append([s for s in X.columns if "MaritalStatus" in str(s)])
    plaf.groups.append([s for s in X.columns if "WorkClass" in str(s)])

    preds = GeCo.score(classifier, X, desired_class, extra_col=0)

    checked = 0
    index = 0

    # failed to generated rules
    fail_generated_n = 0
    fail_generated_i = 0
    fail_generated_m = 0
    fail_generated_im = 0

    # cardinality
    cardinality_n = []
    cardinality_i = []
    cardinality_m = []
    cardinality_im = []
    cardinality_all = []

    success_verified_n = 0
    success_verified_i = 0
    success_verified_m = 0
    success_verified_im = 0
    total_verified_n = 0
    total_verified_i = 0
    total_verified_m = 0
    total_verified_im = 0

    # efficiency -- runtime and generations
    times_n = []
    times_i = []
    times_m = []
    times_im = []
    times_all = []

    generation_n = []
    generation_i = []
    generation_m = []
    generation_im = []
    generation_all = []

    while checked < 500:
        if index >= len(preds):
            break

        pred_val = preds[index]
        cur_idx = index
        index += 1

        if pred_val >= 0.5:
            continue

        n_success = False
        i_success = False
        m_success = False
        im_success = False

        print(f" num checked {checked}, current index{cur_idx}")

        orig_instance = X.iloc[cur_idx, :]

        # only geco
        start_time = time.perf_counter()
        results = geco_for_rule(orig_instance, X, classifier, plaf, desired_class=desired_class)
        elapsed = time.perf_counter() - start_time

        if checked < 20:
            print(elapsed)

        rule = results[0]
        num_exp = results[1]

        if num_exp == -1:
            print("skipped")
            continue

        cardinality_all.append(np.sum(rule.bit_representation))
        times_all.append(elapsed)
        generation_all.append(num_exp)

        # n
        if checked < 20: print("n")
        start_time = time.perf_counter()
        results = generate_rules(orig_instance, X, classifier, plaf, geco_initial=False, geco_mutation=False, desired_class=desired_class)
        elapsed = time.perf_counter() - start_time
        rules = results[0]
        generation = results[1]
        if checked < 20: print(elapsed)

        if not rules or rules[0].precision < 1:
            fail_generated_n += 1
            print(f"fail generate for n at {cur_idx}")
        else:
            total_verified_n += 1
            if geco_verify(rules, orig_instance, plaf, X, classifier, desired_class):
                success_verified_n += 1
                cardinality_n.append(np.sum(rules[0].bit_representation))
                times_n.append(elapsed)
                generation_n.append(generation)
                n_success = True
            else:
                print(f"geco_verify fail for n at {cur_idx}")

        # i
        if checked < 20: print("i")
        start_time = time.perf_counter()
        results = generate_rules(orig_instance, X, classifier, plaf, geco_initial=True, geco_mutation=False, desired_class=desired_class)
        elapsed = time.perf_counter() - start_time
        rules = results[0]
        generation = results[1]
        if checked < 20: print(elapsed)

        if not rules or rules[0].precision < 1:
            fail_generated_i += 1
            print(f"fail generate for i at {cur_idx}")
        else:
            total_verified_i += 1
            if geco_verify(rules, orig_instance, plaf, X, classifier, desired_class):
                success_verified_i += 1
                cardinality_i.append(np.sum(rules[0].bit_representation))
                times_i.append(elapsed)
                generation_i.append(generation)
                i_success = True
            else:
                print(f"geco_verify fail for i at {cur_idx}")

        # m
        if checked < 20: print("m")
        start_time = time.perf_counter()
        results = generate_rules(orig_instance, X, classifier, plaf, geco_initial=False, geco_mutation=True, desired_class=desired_class)
        elapsed = time.perf_counter() - start_time
        rules = results[0]
        generation = results[1]
        if checked < 20: print(elapsed)

        if not rules or rules[0].precision < 1:
            fail_generated_m += 1
            print(f"fail generate for m at {cur_idx}")
        else:
            total_verified_m += 1
            if geco_verify(rules, orig_instance, plaf, X, classifier, desired_class):
                success_verified_m += 1
                cardinality_m.append(np.sum(rules[0].bit_representation))
                times_m.append(elapsed)
                generation_m.append(generation)
                m_success = True
            else:
                print(f"geco_verify fail for m at {cur_idx}")

        # im
        if checked < 20: print("im")
        start_time = time.perf_counter()
        results = generate_rules(orig_instance, X, classifier, plaf, geco_initial=True, geco_mutation=True, desired_class=desired_class)
        elapsed = time.perf_counter() - start_time
        rules = results[0]
        generation = results[1]
        if checked < 20: print(elapsed)

        if not rules or rules[0].precision < 1:
            fail_generated_im += 1
            print(f"fail generate for im at {cur_idx}")
        else:
            total_verified_im += 1
            if geco_verify(rules, orig_instance, plaf, X, classifier, desired_class):
                success_verified_im += 1
                cardinality_im.append(np.sum(rules[0].bit_representation))
                times_im.append(elapsed)
                generation_im.append(generation)
                im_success = True
            else:
                print(f"geco_verify fail for im at {cur_idx}")

        checked += 1

        if not (n_success and i_success and m_success and im_success):
            if n_success:
                cardinality_n.pop()
                times_n.pop()
                generation_n.pop()
            if i_success:
                cardinality_i.pop()
                times_i.pop()
                generation_i.pop()
            if m_success:
                cardinality_m.pop()
                times_m.pop()
                generation_m.pop()
            if im_success:
                cardinality_im.pop()
                times_im.pop()
                generation_im.pop()
            cardinality_all.pop()
            times_all.pop()
            generation_all.pop()
        elif checked < 10:
            print([fail_generated_n, fail_generated_i, fail_generated_m, fail_generated_im])
            print([success_verified_n, success_verified_i, success_verified_m, success_verified_im])
            print([cardinality_n[-1], cardinality_i[-1], cardinality_m[-1], cardinality_im[-1], cardinality_all[-1]])

        if checked % 10 == 0:
            fail_generates = [fail_generated_n, fail_generated_i, fail_generated_m, fail_generated_im]
            cardinalities = [cardinality_n, cardinality_i, cardinality_m, cardinality_im, cardinality_all]
            success_verifieds = [success_verified_n, success_verified_i, success_verified_m, success_verified_im]
            total_verifieds = [total_verified_n, total_verified_i, total_verified_m, total_verified_im]
            times = [times_n, times_i, times_m, times_im, times_all]
            generations = [generation_n, generation_i, generation_m, generation_im, generation_all]

            file_naive = f"rule_based_model/scripts/results/adult_{checked}_2.jld"
            # Using joblib for Python equivalent of JLD
            data_to_save = {
                "fail_generates": fail_generates,
                "cardinalities": cardinalities,
                "total_verifieds": total_verifieds,
                "success_verifieds": success_verifieds,
                "times": times,
                "generations": generations
            }
            try:
                import os
                os.makedirs(os.path.dirname(file_naive), exist_ok=True)
                joblib.dump(data_to_save, file_naive)
                print(f"save {checked}")
            except Exception as e:
                print(f"Failed to save at {checked}: {e}")

    fail_generates = [fail_generated_n, fail_generated_i, fail_generated_m, fail_generated_im]
    cardinalities = [cardinality_n, cardinality_i, cardinality_m, cardinality_im, cardinality_all]
    total_verifieds = [total_verified_n, total_verified_i, total_verified_m, total_verified_im]
    success_verifieds = [success_verified_n, success_verified_i, success_verified_m, success_verified_im]
    times = [times_n, times_i, times_m, times_im, times_all]
    generations = [generation_n, generation_i, generation_m, generation_im, generation_all]

    file_naive = "rule_based_model/scripts/results/adult_2.jld"
    data_to_save = {
        "fail_generates": fail_generates,
        "cardinalities": cardinalities,
        "total_verifieds": total_verifieds,
        "success_verifieds": success_verifieds,
        "times": times,
        "generations": generations
    }
    try:
        import os
        os.makedirs(os.path.dirname(file_naive), exist_ok=True)
        joblib.dump(data_to_save, file_naive)
    except:
        pass

    print("finished")


### Credit Data

#### compare_credit.jl

In [ ]:
import pandas as pd
import numpy as np
import pickle
import time
import random
import math
import copy
import os
from sklearn.ensemble import RandomForestClassifier

# ==============================================================================
# GeCo Logic
# ==============================================================================

class Constraint:
    def __init__(self, features, func):
        self.features = features
        self.func = func # func(orig_instance, *feature_values)

class Implication:
    def __init__(self, condition, consequence, condFeatures, conseqFeatures):
        self.condition = condition # condition(orig, cf) -> bool
        self.consequence = consequence # consequence(orig, cf_vals) -> cf_vals
        self.condFeatures = condFeatures
        self.conseqFeatures = conseqFeatures

class PLAFProgram:
    def __init__(self):
        self.groups = []
        self.constraints = []
        self.implications = []

    def add_constraint(self, features, func):
        self.constraints.append(Constraint(features, func))

    def add_implication(self, condition, consequence, condFeatures, conseqFeatures):
        self.implications.append(Implication(condition, consequence, condFeatures, conseqFeatures))

def initPLAF():
    return PLAFProgram()

def score(classifier, data, desired_class=1, extra_col=0):
    """
    GeCo.score. Returns the prediction probability for the desired class.
    """
    if hasattr(classifier, 'predict_proba'):
        # Scikit-learn style
        return classifier.predict_proba(data)[:, desired_class]
    else:
        # Fallback/Other
        return classifier.predict(data)

class FeatureGroup:
    def __init__(self, features, domain):
        self.features = features
        self.domain = domain # List of possible values (as DataFrames or arrays)

def feasibleSpace(data, orig_instance, program):
    """
    Computes the feasible space for each feature group based on constraints.
    """
    feasible_space = {}
    for feature in data.columns:
        domain = data[feature].unique()
        # Filter domain based on constraints involving only this feature
        # This is a simplification of the full PLAF grounder
        valid_values = []
        for val in domain:
            is_valid = True
            for constr in program.constraints:
                if len(constr.features) == 1 and constr.features[0] == feature:
                    if not constr.func(orig_instance, val):
                        is_valid = False
                        break
            if is_valid:
                valid_values.append(val)
        feasible_space[feature] = np.array(valid_values)
    return feasible_space

def explain(orig_instance, data, program, classifier, desired_class=1, k=100, max_num_generations=10):
    """
    GeCo.explain.
    Returns a DataFrame of counterfactuals.
    """
    feasible_space = feasibleSpace(data, orig_instance, program)

    # Initial population: random samples from feasible space
    pop_size = k
    population = []
    for _ in range(pop_size):
        cf = orig_instance.copy()
        for feature, domain in feasible_space.items():
            if len(domain) > 0:
                cf[feature] = random.choice(domain)
        population.append(cf)

    population_df = pd.DataFrame(population)

    for gen in range(max_num_generations):
        # Score
        preds = score(classifier, population_df, desired_class)
        population_df['score'] = preds
        population_df['outc'] = preds >= 0.5

        # Sort and select top k
        population_df = population_df.sort_values(by='score', ascending=False).head(k)

        # Termination check (if any reached target)
        if population_df['outc'].any():
            pass # Keep going to improve distance, but here we simplify

        # Crossover & Mutation
        offspring = []
        for _ in range(k):
            # Crossover
            p1 = population_df.iloc[random.randint(0, len(population_df)-1)]
            p2 = population_df.iloc[random.randint(0, len(population_df)-1)]
            child = p1.copy()
            for feature in data.columns:
                if random.random() > 0.5:
                    child[feature] = p2[feature]

            # Mutation
            for feature in data.columns:
                if random.random() < 0.1: # mutation rate
                    if len(feasible_space[feature]) > 0:
                        child[feature] = random.choice(feasible_space[feature])
            offspring.append(child)

        population_df = pd.concat([population_df, pd.DataFrame(offspring)], ignore_index=True)

    # Final scoring and sorting
    preds = score(classifier, population_df.drop(columns=['score', 'outc'], errors='ignore'), desired_class)
    population_df['score'] = preds
    population_df['outc'] = preds >= 0.5
    population_df = population_df.sort_values(by='score', ascending=False)

    return population_df, len(population_df), max_num_generations

# ==============================================================================
# GeneticCF Logic
# ==============================================================================

class RuleFeatureGroup:
    def __init__(self, features, indexes, categorical):
        self.features = features
        self.indexes = indexes
        self.categorical = categorical

class RuleComponent:
    def __init__(self, group, direction, index, group_index):
        self.group = group
        self.direction = direction # True for >=, False for <=
        self.index = index
        self.group_index = group_index

class Rule:
    def __init__(self, component_num):
        self.rule_components = []
        self.bit_representation = [False] * component_num
        self.score = 0.0
        self.precision = 0.0
        self.generation = 0
        self.geco_verified = False
        self.geco_generated = False

class ActionComponent:
    def __init__(self, index, direction):
        self.index = index
        self.direction = direction # 1 -> smaller, 2 -> larger (mapped from Julia)

class Action:
    def __init__(self, components):
        self.components = components

def initRuleFeatureGroups(prog, data):
    groups = []
    fidx_groupidx = [0] * data.shape[1]
    included_features = set()

    col_names = list(data.columns)

    for group_features in prog.groups:
        indexes = []
        for feature in group_features:
            idx = col_names.index(feature)
            included_features.add(feature)
            indexes.append(idx)
        groups.append(RuleFeatureGroup(list(group_features), indexes, True))

    for idx, feature in enumerate(col_names):
        if feature not in included_features:
            is_categorical = data[feature].dtype == 'category' or data[feature].dtype == 'object'
            groups.append(RuleFeatureGroup([feature], [idx], is_categorical))

    for gidx, group in enumerate(groups):
        for fidx in group.indexes:
            fidx_groupidx[fidx] = gidx + 1 # 1-based to match Julia

    return groups, fidx_groupidx

def initailComponents(groups):
    components = []
    for idx, group in enumerate(groups):
        group_idx = idx + 1
        # Julia: 2*idx-1, 2*idx. Python: 0-indexed bits
        components.append(RuleComponent(group, True, 2*idx, group_idx))
        components.append(RuleComponent(group, False, 2*idx + 1, group_idx))
    return components

def initialRule(component_num):
    return Rule(component_num)

def initialPopulation_naive(components):
    initial_pop = []
    population_set = set()

    for index, component in enumerate(components):
        rule = initialRule(len(components))
        rule.rule_components.append(component)
        rule.bit_representation[index] = True

        if component.group.categorical:
            if component.direction: # This is the first of the pair
                rule.rule_components.append(components[index+1])
                rule.bit_representation[index+1] = True
            else:
                continue

        initial_pop.append(rule)
        population_set.add(tuple(rule.bit_representation))

    return initial_pop, population_set

def initialSpace(orig_instance, data):
    space = []
    for feature in data.columns:
        feature_space = []
        all_values = data[feature].values
        feature_space.append(all_values[all_values >= orig_instance[feature]])
        feature_space.append(all_values[all_values <= orig_instance[feature]])
        space.append(feature_space)
    return space

def sample_data(data, rule, orig_instance, space, number=1000):
    rows = np.random.choice(len(data), number)
    samples = data.iloc[rows].copy()

    # Sort rule components by group_index for deterministic logic
    sorted_components = sorted(rule.rule_components, key=lambda x: x.index)

    i = 0
    while i < len(sorted_components):
        component = sorted_components[i]
        group_idx = component.group_index

        # Check if both directions exist for this group
        has_dual = False
        if i + 1 < len(sorted_components) and sorted_components[i+1].group_index == group_idx:
            has_dual = True

        direction = 2 if has_dual else (1 if not component.direction else 0)

        for fidx, feature in zip(component.group.indexes, component.group.features):
            if direction == 2: # Exact match
                samples[feature] = orig_instance[feature]
            elif direction == 1: # <=
                mask = samples[feature] > orig_instance[feature]
                violates_count = mask.sum()
                if violates_count > 0:
                    replaces = np.random.choice(space[fidx][1], violates_count)
                    samples.loc[mask, feature] = replaces
            else: # >=
                mask = samples[feature] < orig_instance[feature]
                violates_count = mask.sum()
                if violates_count > 0:
                    replaces = np.random.choice(space[fidx][0], violates_count)
                    samples.loc[mask, feature] = replaces

        i += 2 if has_dual else 1

    return samples

def evaluation(data, rule, orig_instance, space, classifier, epslin=0.05, desired_class=1, sample_num=1000):
    samples = sample_data(data, rule, orig_instance, space, number=sample_num)
    preds = score(classifier, samples, desired_class)

    precision = np.sum(preds < 0.5) / len(preds)

    rule.precision = precision
    bit_len = len(rule.bit_representation)
    bit_sum = sum(rule.bit_representation)

    if rule.geco_verified:
        rule.score = 1.0 / bit_len * 0.25 + 0.5 - bit_sum / bit_len * 0.5 + precision * 0.5
    elif precision < 1.0 - epslin:
        if precision < 0.25 and rule.geco_generated:
            precision += bit_sum / bit_len
        rule.score = precision * 0.5
    else:
        rule.score = 0.5 - bit_sum / bit_len * 0.5 + precision * 0.5

def selection(population, data, orig_instance, space, classifier, checked_set, pop_size=10, epslin=0.95, desired_class=1, sample_num=1000):
    for rule in population:
        if rule.score == 0:
            evaluation(data, rule, orig_instance, space, classifier, epslin=epslin, desired_class=desired_class, sample_num=sample_num)

    population.sort(key=lambda x: x.score, reverse=True)

    if len(population) > pop_size:
        for i in range(pop_size, len(population)):
            if population[i].score < 0.5:
                break
            else:
                bits = tuple(population[i].bit_representation)
                if bits in checked_set:
                    checked_set.remove(bits)
        del population[pop_size:]

def mutation(population, checked_set, components, generation, max_num_samples=3, pop_size=20):
    row_num = min(len(population), pop_size)
    col_num = len(components)

    for row_index in range(row_num):
        rule = population[row_index]
        possible_components = [i for i in range(col_num) if not rule.bit_representation[i]]

        changes = random.sample(possible_components, min(len(possible_components), max_num_samples))
        for index in changes:
            mutated_rule = copy.deepcopy(rule)
            mutated_rule.generation = generation
            mutated_rule.geco_verified = False
            mutated_rule.geco_generated = False
            mutated_rule.score = 0

            comp = components[index]
            if comp.group.categorical:
                group_idx = (index // 2)
                mutated_rule.bit_representation[group_idx * 2] = True
                mutated_rule.rule_components.append(components[group_idx * 2])
                mutated_rule.bit_representation[group_idx * 2 + 1] = True
                mutated_rule.rule_components.append(components[group_idx * 2 + 1])
            else:
                mutated_rule.bit_representation[index] = True
                mutated_rule.rule_components.append(comp)

            bits = tuple(mutated_rule.bit_representation)
            if bits not in checked_set:
                mutated_rule.rule_components.sort(key=lambda x: x.index)
                population.append(mutated_rule)
                checked_set.add(bits)

def crossover(population, checked_set, components, generation, pop_size=20):
    row_num = min(len(population), pop_size)
    col_num = len(components)
    for p1_idx in range(row_num):
        for p2_idx in range(p1_idx + 1, row_num):
            for i in range(2): # Try creating two children
                parent1 = population[p1_idx]
                parent2 = population[p2_idx]

                child = copy.deepcopy(parent1 if i == 0 else parent2)
                other = parent2 if i == 0 else parent1

                child.score = 0
                child.generation = generation
                child.geco_verified = False
                child.geco_generated = False

                diff = [j for j in range(col_num) if other.bit_representation[j] and not child.bit_representation[j]]

                if diff:
                    add_comp_idx = random.choice(diff)
                    comp = components[add_comp_idx]
                    if comp.group.categorical:
                        g_idx = add_comp_idx // 2
                        child.bit_representation[g_idx * 2] = True
                        child.rule_components.append(components[g_idx * 2])
                        child.bit_representation[g_idx * 2 + 1] = True
                        child.rule_components.append(components[g_idx * 2 + 1])
                    else:
                        child.bit_representation[add_comp_idx] = True
                        child.rule_components.append(comp)

                    bits = tuple(child.bit_representation)
                    if bits not in checked_set:
                        child.rule_components.sort(key=lambda x: x.index)
                        population.append(child)
                        checked_set.add(bits)

def fetch_feature_actions(counterfactuals, orig_instance):
    res = []
    for idx in range(len(counterfactuals)):
        cf = counterfactuals.iloc[idx]
        if not cf['outc']: continue

        action_comps = []
        for fidx, feature in enumerate(orig_instance.index):
            delta = cf[feature] - orig_instance[feature]
            if delta != 0:
                direction = 2 if delta > 0 else 1
                action_comps.append(ActionComponent(fidx, direction))
        res.append(Action(action_comps))
    return res

def actions_reduced_groups(feature_actions, fidx_gidx):
    group_actions = []
    for f_act in feature_actions:
        comps = []
        g_idxs = set()
        for c in f_act.components:
            gidx = fidx_gidx[c.index]
            if gidx not in g_idxs:
                g_idxs.add(gidx)
                comps.append(ActionComponent(gidx, c.direction))
        group_actions.append(Action(comps))

    group_actions.sort(key=lambda x: len(x.components))

    # Reduction logic
    aidx = 1
    while aidx < len(group_actions):
        cur = group_actions[aidx]
        covered = False
        for prev_idx in range(aidx):
            prev = group_actions[prev_idx]
            # check if prev is subset of cur
            is_subset = True
            prev_comps = {(c.index, c.direction) for c in prev.components}
            cur_comps = {(c.index, c.direction) for c in cur.components}
            if not prev_comps.issubset(cur_comps):
                is_subset = False
            if is_subset:
                covered = True
                break
        if covered:
            group_actions.pop(aidx)
        else:
            aidx += 1
    return group_actions

def action_to_rule(action_components, components, generation):
    rule = initialRule(len(components))
    rule.generation = generation
    for act_comp in action_components:
        cindex = act_comp.index * 2 - 1 # Adjusted from Julia logic
        if components[cindex].group.categorical:
            if rule.bit_representation[cindex]: continue
            rule.bit_representation[cindex] = True
            rule.bit_representation[cindex - 1] = True
            rule.rule_components.append(components[cindex])
            rule.rule_components.append(components[cindex - 1])
        else:
            real_idx = cindex if act_comp.direction == 2 else cindex - 1
            if rule.bit_representation[real_idx]: continue
            rule.bit_representation[real_idx] = True
            rule.rule_components.append(components[real_idx])
    rule.geco_generated = True
    return rule

def backtrace_geco(actions, index, action_components, components, population, checked_set, generation, max_action=5):
    if index > min(len(actions), max_action):
        rule = action_to_rule(action_components, components, generation)
        bits = tuple(rule.bit_representation)
        if bits not in checked_set:
            rule.rule_components.sort(key=lambda x: x.index)
            population.append(rule)
            checked_set.add(bits)
        return

    for act_comp in actions[index-1].components:
        action_components.append(act_comp)
        backtrace_geco(actions, index + 1, action_components, components, population, checked_set, generation)
        action_components.pop()

def initialPopulationWithGeco(data, orig_instance, plaf_program, classifier, fidx_gidx, components, population, checked_set, desired_class=1):
    explains, _, _ = explain(orig_instance, data, plaf_program, classifier, desired_class=desired_class)
    feature_actions = fetch_feature_actions(explains, orig_instance)
    group_actions = actions_reduced_groups(feature_actions, fidx_gidx)
    backtrace_geco(group_actions, 1, [], components, population, checked_set, 0)
    return population, checked_set

def rule_to_constrain(rule, plaf_program, orig_instance):
    plaf = copy.deepcopy(plaf_program)
    current_components = []

    sorted_comps = sorted(rule.rule_components, key=lambda x: x.index)

    i = 0
    while i < len(sorted_comps):
        comp = sorted_comps[i]
        g_idx = comp.group_index

        has_dual = False
        if i + 1 < len(sorted_comps) and sorted_comps[i+1].group_index == g_idx:
            has_dual = True

        direction = 2 if has_dual else (0 if comp.direction else 1)
        if direction == 2:
            current_components.append(ActionComponent(g_idx, 1))
            current_components.append(ActionComponent(g_idx, 2))
        elif direction == 0:
            current_components.append(ActionComponent(g_idx, 1))
        else:
            current_components.append(ActionComponent(g_idx, 2))

        for feature in comp.group.features:
            val = orig_instance[feature]
            if direction == 2:
                plaf.add_constraint([feature], lambda orig, cf_val, v=val: cf_val == v)
            elif direction == 1:
                plaf.add_constraint([feature], lambda orig, cf_val, v=val: cf_val <= v)
            else:
                plaf.add_constraint([feature], lambda orig, cf_val, v=val: cf_val >= v)

        i += 2 if has_dual else 1
    return plaf, current_components

def mutation_geco(population, checked_set, data, orig_instance, classifier, plaf_program, fidx_gidx, components, generation, desired_class=1, check_size=10):
    row_num = min(len(population), check_size)
    idx = 0
    changed = False

    while row_num > 0:
        row_num -= 1
        if idx >= len(population): break
        rule = population[idx]
        if rule.geco_verified:
            idx += 1
            continue

        p, action_components = rule_to_constrain(rule, plaf_program, orig_instance)
        explains, _, _ = explain(orig_instance, data, p, classifier, desired_class=desired_class)

        if len(explains) == 0 or not explains.iloc[0]['outc']:
            rule.geco_verified = True
            idx += 1
            continue

        rule.geco_verified = False
        cur_pop_size = len(population)
        changed = True

        feature_acts = fetch_feature_actions(explains, orig_instance)
        group_acts = actions_reduced_groups(feature_acts, fidx_gidx)
        backtrace_geco(group_acts, 1, action_components, components, population, checked_set, generation)

        if len(population) > cur_pop_size or len(population) > 2:
            population.pop(idx)
        else:
            idx += 1
    return changed

def check_converge(population, generation, geco_mutation):
    if not population: return True
    if geco_mutation and population[0].geco_verified and population[0].generation + 3 <= generation:
        return True

    for i, rule in enumerate(population):
        if rule.generation + 1 >= generation:
            return False
        if i >= 2 and population[0].geco_verified:
            return True
        if i >= 4:
            return True
    return True

def generate_rules(orig_instance, data, classifier, plaf_program, geco_initial=False, geco_mutation=False, epslin=0.0, max_num_population=50, desired_class=1, max_generation=50, sample_num=1000, geco_check_size=5):
    groups, fidx_gidx = initRuleFeatureGroups(plaf_program, data)
    components = initailComponents(groups)

    population, checked_set = initialPopulation_naive(components)

    if geco_initial:
        population, checked_set = initialPopulationWithGeco(data, orig_instance, plaf_program, classifier, fidx_gidx, components, population, checked_set, desired_class=desired_class)

    space = initialSpace(orig_instance, data)
    selection(population, data, orig_instance, space, classifier, checked_set, pop_size=max_num_population, epslin=epslin, desired_class=desired_class, sample_num=sample_num)

    generation = 0
    converge = False
    last_geco = 0

    while generation < max_generation and (not converge or not population or population[0].precision < 1 - epslin):
        generation += 1

        if geco_mutation and generation - last_geco > 3:
            last_geco = generation
            mutation_geco(population, checked_set, data, orig_instance, classifier, plaf_program, fidx_gidx, components, generation, desired_class=desired_class, check_size=geco_check_size)

        ori_pop_size = len(population)
        crossover(population, checked_set, components, generation, pop_size=max_num_population)
        mutation(population, checked_set, components, generation, pop_size=min(ori_pop_size, max_generation))
        selection(population, data, orig_instance, space, classifier, checked_set, pop_size=max_num_population, epslin=epslin, desired_class=desired_class, sample_num=sample_num)

        converge = check_converge(population, generation, geco_mutation)
        if geco_mutation and converge:
            if population[0].geco_verified:
                population[0].geco_verified = False
                changed = mutation_geco(population, checked_set, data, orig_instance, classifier, plaf_program, fidx_gidx, components, generation, desired_class=desired_class, check_size=1)
                selection(population, data, orig_instance, space, classifier, checked_set, pop_size=max_num_population, epslin=epslin, desired_class=desired_class, sample_num=sample_num)
                if changed: converge = False
            else:
                converge = False

    return population, generation

def geco_for_rule(orig_instance, data, classifier, plaf_program, desired_class=1):
    groups, fidx_gidx = initRuleFeatureGroups(plaf_program, data)
    components = initailComponents(groups)

    initial_pop = []
    population_set = set()
    population, checked_set = initialPopulationWithGeco(data, orig_instance, plaf_program, classifier, fidx_gidx, components, initial_pop, population_set, desired_class=desired_class)

    population.sort(key=lambda x: len(x.rule_components), reverse=True)
    num_exp = 1

    while True:
        if not population: return None, -1
        rule = population.pop()
        num_exp += 1
        p_cur, action_components = rule_to_constrain(rule, plaf_program, orig_instance)
        explains, _, _ = explain(orig_instance, data, p_cur, classifier, desired_class=desired_class)

        if len(explains) == 0 or not explains.iloc[0]['outc']:
            return rule, num_exp

        feature_acts = fetch_feature_actions(explains, orig_instance)
        group_acts = actions_reduced_groups(feature_acts, fidx_gidx)
        backtrace_geco(group_acts, 1, action_components, components, population, checked_set, 0)

        if not population: return rule, -1
        population.sort(key=lambda x: len(x.rule_components), reverse=True)

# ==============================================================================
# credit_setup_MACE.jl and compare_credit.jl
# ==============================================================================

def geco_verify(rules, orig_instance, X, classifier):
    # verify using geco
    rule = rules[0]
    # create the constraints for plaf by the rule
    p, _ = rule_to_constrain(rule, initPLAF(), orig_instance)
    geco_explanation, _, _ = explain(orig_instance, X, p, classifier)
    if len(geco_explanation) != 0 and geco_explanation.iloc[0]['outc']:
        return False
    return True

def run_exp(X, classifier):
    plaf = initPLAF()
    # Setup constraints from credit_constraints_MACE.jl
    plaf.add_constraint(['isMale'], lambda orig, val: val == orig['isMale'])
    plaf.add_constraint(['isMarried'], lambda orig, val: val == orig['isMarried'])
    plaf.add_constraint(['AgeGroup'], lambda orig, val: val >= orig['AgeGroup'])
    plaf.add_constraint(['EducationLevel'], lambda orig, val: val >= orig['EducationLevel'])
    plaf.add_constraint(['HasHistoryOfOverduePayments'], lambda orig, val: val >= orig['HasHistoryOfOverduePayments'])
    plaf.add_constraint(['TotalOverdueCounts'], lambda orig, val: val >= orig['TotalOverdueCounts'])
    plaf.add_constraint(['TotalMonthsOverdue'], lambda orig, val: val >= orig['TotalMonthsOverdue'])
    # Implication logic simplified as constraint
    def education_age_constraint(orig, cf_vals):
        # features: EducationLevel, AgeGroup
        if cf_vals[0] > orig['EducationLevel'] + 1 and orig['AgeGroup'] < 2:
            return cf_vals[1] == 2
        return True
    plaf.add_constraint(['EducationLevel', 'AgeGroup'], education_age_constraint)

    preds = score(classifier, X, 1, extra_col=0)

    checked = 0
    index = -1

    fail_generated_n = fail_generated_i = fail_generated_m = fail_generated_im = 0

    cardinality_n, cardinality_i, cardinality_m, cardinality_im, cardinality_all = [], [], [], [], []
    success_verified_n = success_verified_i = success_verified_m = success_verified_im = 0
    total_verified_n = total_verified_i = total_verified_m = total_verified_im = 0
    times_n, times_i, times_m, times_im, times_all = [], [], [], [], []
    generation_n, generation_i, generation_m, generation_im, generation_all = [], [], [], [], []

    while checked < 3:
        index += 1
        if index >= len(preds): break
        if preds[index] >= 0.5:
            continue

        n_success = i_success = m_success = im_success = False
        print(f" num checked {checked}, current index {index}")
        orig_instance = X.iloc[index]

        # only geco
        start = time.time()
        rule, num_exp = geco_for_rule(orig_instance, X, classifier, plaf)
        elapsed = time.time() - start

        if num_exp == -1:
            print("skipped")
            continue

        cardinality_all.append(sum(rule.bit_representation))
        times_all.append(elapsed)
        generation_all.append(num_exp)

        # n
        start = time.time()
        rules, gen = generate_rules(orig_instance, X, classifier, plaf, geco_initial=False, geco_mutation=False)
        elapsed = time.time() - start
        if not rules or rules[0].precision < 1:
            fail_generated_n += 1
            print(f"fail generate for n at {index}")
        else:
            total_verified_n += 1
            if geco_verify(rules, orig_instance, X, classifier):
                success_verified_n += 1
                cardinality_n.append(sum(rules[0].bit_representation))
                times_n.append(elapsed)
                generation_n.append(gen)
                n_success = True
            else: print(f"geco_verify fail for n at {index}")

        # i
        start = time.time()
        rules, gen = generate_rules(orig_instance, X, classifier, plaf, geco_initial=True, geco_mutation=False)
        elapsed = time.time() - start
        if not rules or rules[0].precision < 1:
            fail_generated_i += 1
            print(f"fail generate for i at {index}")
        else:
            total_verified_i += 1
            if geco_verify(rules, orig_instance, X, classifier):
                success_verified_i += 1
                cardinality_i.append(sum(rules[0].bit_representation))
                times_i.append(elapsed)
                generation_i.append(gen)
                i_success = True
            else: print(f"geco_verify fail for i at {index}")

        # m
        start = time.time()
        rules, gen = generate_rules(orig_instance, X, classifier, plaf, geco_initial=False, geco_mutation=True)
        elapsed = time.time() - start
        if not rules or rules[0].precision < 1:
            fail_generated_m += 1
            print(f"fail generate for m at {index}")
        else:
            total_verified_m += 1
            if geco_verify(rules, orig_instance, X, classifier):
                success_verified_m += 1
                cardinality_m.append(sum(rules[0].bit_representation))
                times_m.append(elapsed)
                generation_m.append(gen)
                m_success = True
            else: print(f"geco_verify fail for m at {index}")

        # im
        start = time.time()
        rules, gen = generate_rules(orig_instance, X, classifier, plaf, geco_initial=True, geco_mutation=True)
        elapsed = time.time() - start
        if not rules or rules[0].precision < 1:
            fail_generated_im += 1
            print(f"fail generate for im at {index}")
        else:
            total_verified_im += 1
            if geco_verify(rules, orig_instance, X, classifier):
                success_verified_im += 1
                cardinality_im.append(sum(rules[0].bit_representation))
                times_im.append(elapsed)
                generation_im.append(gen)
                im_success = True
            else: print(f"geco_verify fail for im at {index}")

        checked += 1
        if not (n_success and i_success and m_success and im_success):
            if n_success: cardinality_n.pop(); times_n.pop(); generation_n.pop()
            if i_success: cardinality_i.pop(); times_i.pop(); generation_i.pop()
            if m_success: cardinality_m.pop(); times_m.pop(); generation_m.pop()
            if im_success: cardinality_im.pop(); times_im.pop(); generation_im.pop()
            cardinality_all.pop(); times_all.pop(); generation_all.pop()
        elif checked < 10:
            print([fail_generated_n, fail_generated_i, fail_generated_m, fail_generated_im])
            print([success_verified_n, success_verified_i, success_verified_m, success_verified_im])
            print([cardinality_n[-1], cardinality_i[-1], cardinality_m[-1], cardinality_im[-1], cardinality_all[-1]])

        if checked % 10 == 0:
            results = {
                "fail_generates": [fail_generated_n, fail_generated_i, fail_generated_m, fail_generated_im],
                "cardinalities": [cardinality_n, cardinality_i, cardinality_m, cardinality_im, cardinality_all],
                "success_verifieds": [success_verified_n, success_verified_i, success_verified_m, success_verified_im],
                "total_verifieds": [total_verified_n, total_verified_i, total_verified_m, total_verified_im],
                "times": [times_n, times_i, times_m, times_im, times_all],
                "generations": [generation_n, generation_i, generation_m, generation_im, generation_all]
            }
            with open(f"credit_{checked}.pkl", "wb") as f:
                pickle.dump(results, f)
            print(f"save {checked}")

    final_results = {
        "fail_generates": [fail_generated_n, fail_generated_i, fail_generated_m, fail_generated_im],
        "cardinalities": [cardinality_n, cardinality_i, cardinality_m, cardinality_im, cardinality_all],
        "success_verifieds": [success_verified_n, success_verified_i, success_verified_m, success_verified_im],
        "total_verifieds": [total_verified_n, total_verified_i, total_verified_m, total_verified_im],
        "times": [times_n, times_i, times_m, times_im, times_all],
        "generations": [generation_n, generation_i, generation_m, generation_im, generation_all]
    }
    with open("credit.pkl", "wb") as f:
        pickle.dump(final_results, f)
    print("finished")

# ==============================================================================
# compare_credit.j;
# ==============================================================================

if __name__ == "__main__":
    # Load data
    try:
        X = pd.read_csv("credit_processed.csv")
        # Preprocessing similar to setup script
        X = X.drop(columns=['NoDefaultNextMonth'], errors='ignore')
    except:
        # Generate dummy data for demonstration if CSV missing
        print("Data file not found. Generating dummy data for demonstration.")
        cols = ['isMale', 'isMarried', 'AgeGroup', 'EducationLevel',
                'MaxBillAmountOverLast6Months', 'MaxPaymentAmountOverLast6Months',
                'MonthsWithZeroBalanceOverLast6Months', 'MonthsWithLowSpendingOverLast6Months',
                'MonthsWithHighSpendingOverLast6Months', 'MostRecentBillAmount',
                'MostRecentPaymentAmount', 'TotalOverdueCounts', 'TotalMonthsOverdue',
                'HasHistoryOfOverduePayments']
        X = pd.DataFrame(np.random.rand(200, len(cols)), columns=cols)
        X['isMale'] = np.random.randint(0, 2, 200)
        X['AgeGroup'] = np.random.randint(1, 4, 200)

    # Load classifier
    try:
        with open("credit_model.pickle", "rb") as f:
            classifier = pickle.load(f)
    except:
        # Dummy classifier for demonstration if pickle missing
        print("Model file not found. Creating a dummy classifier.")
        classifier = RandomForestClassifier(n_estimators=10)
        y_dummy = np.random.randint(0, 2, len(X))
        classifier.fit(X, y_dummy)

    run_exp(X, classifier)


### Fico Data

#### classifier.jl -> classifier.py

For the fico database there was a classifier file that I think was used to decern some of the categorical data. Here is the code from the `classifier.jl` file "de julia-ized", i.e. I converted the code from julia to python. A bit difficult but it was doable. I had to remove the end keyword and also used def instead of function.

In [ ]:
## the classifier for the fico dataset
import pandas as pd
import numpy as np
import math

## transform the entity to vector (bucketize)
# convert an entity to vector of sublayers
def entity_to_vectors(entity):
    vect = [0] * 23
    vect[0] =  feature_to_vector(entity, "ExternalRiskEstimate")
    vect[1] =  feature_to_vector(entity, "MSinceOldestTradeOpen")
    vect[2] =  feature_to_vector(entity, "MSinceMostRecentTradeOpen")
    vect[3] =  feature_to_vector(entity, "AverageMInFile")
    vect[4] =  feature_to_vector(entity, "NumSatisfactoryTrades")
    vect[5] =  feature_to_vector(entity, "NumTrades60Ever2DerogPubRec")
    vect[6] =  feature_to_vector(entity, "NumTrades90Ever2DerogPubRec")
    vect[7] =  feature_to_vector(entity, "NumTotalTrades")
    vect[8] =  feature_to_vector(entity, "NumTradesOpeninLast12M")
    vect[9] =  feature_to_vector(entity, "PercentTradesNeverDelq")
    vect[10] =  feature_to_vector(entity, "MSinceMostRecentDelq")
    vect[11] =  feature_to_vector(entity, "MaxDelq2PublicRecLast12M")
    vect[12] =  feature_to_vector(entity, "MaxDelqEver")
    vect[13] =  feature_to_vector(entity, "PercentInstallTrades")
    vect[14] =  feature_to_vector(entity, "NetFractionInstallBurden")
    vect[15] =  feature_to_vector(entity, "NumInstallTradesWBalance")
    vect[16] =  feature_to_vector(entity, "MSinceMostRecentInqexcl7days")
    vect[17] =  feature_to_vector(entity, "NumInqLast6M")
    # Note: original Julia used indices 18, 19, 20... up to 23
    vect[18] =  feature_to_vector(entity, "NumInqLast6Mexcl7days")
    vect[19] =  feature_to_vector(entity, "NetFractionRevolvingBurden")
    vect[20] =  feature_to_vector(entity, "NumRevolvingTradesWBalance")
    vect[21] =  feature_to_vector(entity, "NumBank2NatlTradesWHighUtilization")
    vect[22] =  feature_to_vector(entity, "PercentTradesWBalance")
    return vect


## bucketize for features
feature_ranges = {"ExternalRiskEstimate": [64,71,76,81],
                      "MSinceOldestTradeOpen": [92,135,264],
                      "MSinceMostRecentTradeOpen": [20],
                      "AverageMInFile": [49,70,97],
                      "NumSatisfactoryTrades": [3,6,13,22],
                      "NumTrades60Ever2DerogPubRec": [2,3,12,13],
                      "NumTrades90Ever2DerogPubRec": [2,8,10],
                      "PercentTradesNeverDelq": [59,84,89,96],
                      "MSinceMostRecentDelq": [18,33,48],
                      "MaxDelq2PublicRecLast12M": [6,7],
                      "MaxDelqEver": [3],
                      "NumTotalTrades": [1,10,17,28],
                      "NumTradesOpeninLast12M": [3,4,7,12],
                      "PercentInstallTrades": [36,47,58,85],
                      "MSinceMostRecentInqexcl7days": [1,2,9,23],
                      "NumInqLast6M": [2,5,9],
                      "NumInqLast6Mexcl7days": [3],
                      "NetFractionRevolvingBurden": [15,38,73],
                      "NetFractionInstallBurden": [36,71],
                      "NumRevolvingTradesWBalance": [4,5,8,12],
                      "NumInstallTradesWBalance": [3,4,12,14],
                      "NumBank2NatlTradesWHighUtilization": [2,3,4,6],
                      "PercentTradesWBalance": [48,67,74,87]}


# given entity and feature, produce the bucketized index for that pair
def feature_to_vector(entity, feature):
    ranges = feature_ranges[feature]
    val = entity[feature]
    if val < 0:
        if val == -7:
            return len(ranges) + 2
        elif val == -8:
            return len(ranges) + 3
        elif val == -9:
            return len(ranges) + 4
    else:
        for i, r in enumerate(ranges):
            if val < r:
                return i + 1
    return len(ranges) + 1



## Subscore for each sublayer
def external_risk_subscore_from_vector(vect):
    weights = [2.9895622, 2.1651128, 1.4081029, 0.7686735, 0, 0, 0, 1.6943381]
    score = -1.4308699
    return score + weights[vect[0] - 1]

def trade_open_time_subscore_from_vector(vect):
    weights1 = [0.820842027, 0.525120503, 0.245257364, 0.005524848, 0, 0.418318111, 0.435851213]
    weights2 = [0.031074792, 0.006016629, 0, 0, 0.027688067]
    weights3 = [ 1.209930852, 0.694452470, 0.296029824, 0, 0, 0, 0.471490736]
    score = -0.696619002
    return score + weights1[vect[1] - 1] + weights2[vect[2] - 1] + weights3[vect[3] - 1]

def num_sat_trades_subscore_from_vector(vect):
    weights = [2.412574, 1.245278, 6.619963e-01, 2.731984e-01, 5.444148e-09, 0, 0, 4.338848e-01]
    score = -1.954726e-01
    return score + weights[vect[4] - 1]

# high numbers!
def trade_freq_subscore_from_vector(vect):
    weights1 = [2.710260e-04, 9.195886e-01, 9.758620e-01, 1.008107e+01, 9.360290, 0, 0, 3.970360e-01]
    weights2 = [1.514937e-01, 3.139667e-01, 0, 2.422345e-01, 0, 0, 3.095043e-02]
    weights3 = [2.888436e-01, 9.659472e-01, 5.142479e-01, 2.653203e-01, 8.198233e-07, 0, 0, 3.233593e-01]
    weights4 =[8.405069e-06, 3.374686e-01, 4.934466e-01, 8.601860e-01, 9.451724, 0, 0, 1.351433e-01]
    score = -6.480598e-01
    return score + weights1[vect[5] - 1] + weights2[vect[6] - 1] + weights3[vect[7] - 1] + weights4[vect[8] - 1]

def delinquency_subscore_from_vector(vect):
    weights1 = [1.658975, 1.218405, 8.030501e-01, 5.685712e-01, 0, 0, 0, 6.645698e-01]
    weights2 = [4.014945e-01, 2.912651e-01, 5.665418e-02, 0,6.935965e-01, 5.470874e-01, 4.786956e-01]
    weights3 = [1.004642, 5.654694e-01, 0, 0, 0, 2.841047e-01]
    weights4 = [1.378803e-01, 1.101649e-06, 0, 0, 1.051132e-02]
    score = -1.199469
    return score + weights1[vect[9] - 1] + weights2[vect[10] - 1] + weights3[vect[11] - 1] + weights4[vect[12] - 1]

def installment_subscore_from_vector(vect):
    weights1 = [9.059412e-05, 1.292266e-01, 4.680034e-01, 8.117938e-01, 1.954441, 0, 0, 1.281830]
    weights2 = [0, 1.432068e-01, 3.705526e-01, 0, 4.972869e-03, 1.513885e-01]
    weights3 = [1.489759, 1.478176, 1.518328, 0, 9.585058e-01, 0, 1.506442, 5.561296e-01]
    score = -1.750937
    return score + weights1[vect[13] - 1] + weights2[vect[14] - 1] + weights3[vect[15] - 1]

def inquiry_subscore_from_vector(vect):
    weights1 = [1.907737, 1.260966, 1.010585, 8.318137e-01, 0, 1.951357, 0, 1.719356]
    weights2 = [2.413596e-05, 2.251582e-01, 5.400251e-01, 1.255076, 0, 0, 1.061504e-01]
    weights3 = [0, 6.095516e-02, 0, 0, 1.125418e-02]
    score = -1.598351
    return score + weights1[vect[16] - 1] + weights2[vect[17] - 1] + weights3[vect[18] - 1]

def revol_balance_subscore_from_vector(vect):
    weights1 = [0.0001042232, 0.6764476961, 1.3938464180, 2.2581926077, 0, 1.7708134303, 1.0411847907]
    weights2 = [0.0756555085, 0, 0.1175915408, 0.2823307493, 0.4242649887, 0, 0.8756715032, 0.0897134843]
    score = -0.8924856930
    return score + weights1[vect[19] - 1] + weights2[vect[20] - 1]

def utilization_subscore_from_vector(vect):
    weights = [0, 0.8562096, 1.2047649, 1.1635459, 1.4701220, 0, 1.2392294, 0.4800086]
    score = -0.2415871
    return score + weights[vect[21] - 1]

def trade_w_balance_subscore_from_vector(vect):
    weights = [0, 0.5966752, 0.9207121, 1.2749998, 1.8474869, 0, 2.2885183, 1.0606029]
    score = -0.8221922
    return score + weights[vect[22] - 1]

# Sigmoid function
def sigmoid(x):
    return 1 / (1 + math.exp(-1 * x))


# the risk model to returns risks
def risk_model(entity):
    vect = entity_to_vectors(entity)
    subscore = [0.0] * 10
    subscore[0] = sigmoid(external_risk_subscore_from_vector(vect))
    subscore[1] = sigmoid(trade_open_time_subscore_from_vector(vect))
    subscore[2] = sigmoid(num_sat_trades_subscore_from_vector(vect))
    subscore[3] = sigmoid(trade_freq_subscore_from_vector(vect))
    subscore[4] = sigmoid(delinquency_subscore_from_vector(vect))
    subscore[5] = sigmoid(installment_subscore_from_vector(vect))
    subscore[6] = sigmoid(inquiry_subscore_from_vector(vect))
    subscore[7] = sigmoid(revol_balance_subscore_from_vector(vect))
    subscore[8] = sigmoid(utilization_subscore_from_vector(vect))
    subscore[9] = sigmoid(trade_w_balance_subscore_from_vector(vect))
    weights = [1.5671672, 2.5236825, 2.1711503, 0.3323177, 2.5396631, 0.9148520,
               3.0015073, 1.9259728, 0.9864329, 0.2949793]
    score = -8.3843046
    for i in range(len(weights)):
        score += (subscore[i] * weights[i])
    return sigmoid(score)

## the classifier returns good (0) or bad (1)
def classify(entity):
    prediction = risk_model(entity)

    if prediction >= 0.5:
        return 1
    else:
        return 0


#### Fico.jl -> Fico.py
There is also an additional Fico.jl file for another classifier in the GeCo file structure.

In [ ]:
import numpy as np
import pandas as pd
import warnings
from sklearn.base import BaseEstimator

# tree_model = @load FICO_CLASSIFIER pkg=GeneticCounterfactual
class FICO_CLASSIFIER(BaseEstimator):
    """
    MLJModelInterface.
    """
    def __init__(self, lambda_=0.0):
        self.lambda_ = lambda_
        self._clean()

    def _clean(self):
        """
        MMI.clean!(model)
        """
        warning = ""
        if self.lambda_ < 0:
            warning += "Need lambda ≥ 0. Resetting lambda=0. "
            self.lambda_ = 0

        if warning:
            warnings.warn(warning)
        return warning

    def fit(self, X, y=None, verbosity=1):
        """
        MMI.fit(model, verbosity, X, y)
        """
        self.fitresult_ = risk_model
        cache = None
        report = None
        return self.fitresult_, cache, report

    def predict(self, Xnew):
        """
        predict(model::FICO_CLASSIFIER, Xnew)
        """

        # Check if Xnew is a DataFrame or numpy array
        if isinstance(Xnew, pd.DataFrame):
            n_rows = Xnew.shape[0]
            outcomes = np.zeros(n_rows, dtype=float)
            for i in range(n_rows):
                # row is passed to risk_model
                outcomes[i] = 1.0 - risk_model(Xnew.iloc[i])
        else:
            # Assume numpy-like
            n_rows = Xnew.shape[0]
            outcomes = np.zeros(n_rows, dtype=float)
            for i in range(n_rows):
                outcomes[i] = 1.0 - risk_model(Xnew[i])

        return outcomes

def feature_to_vector(val, ranges):
    val = float(val)
    if val < 0.0:
        if val == -7.0:
            return len(ranges) + 2
        elif val == -8.0:
            return len(ranges) + 3
        elif val == -9.0:
            return len(ranges) + 4
    else:
        for i, r in enumerate(ranges):
            if val < float(r):
                # Julia is 1-indexed, so we return i + 1
                return i + 1

    return len(ranges) + 1

def entity_to_vectors(entity):
    # values::Array{Float64,1} = convert(Array{Float64,1}, entity)
    if hasattr(entity, "to_numpy"):
        values = entity.to_numpy().astype(float)
    else:
        values = np.array(entity, dtype=float)

    ExternalRiskEstimate = [64.0, 71.0, 76.0, 81.0]
    MSinceOldestTradeOpen = [92.0, 135.0, 264.0]
    MSinceMostRecentTradeOpen = [20.0]
    AverageMInFile = [49.0, 70.0, 97.0]
    NumSatisfactoryTrades = [3.0, 6.0, 13.0, 22.0]
    NumTrades60Ever2DerogPubRec = [2.0, 3.0, 12.0, 13.0]
    NumTrades90Ever2DerogPubRec = [2.0, 8.0, 10.0]
    PercentTradesNeverDelq = [59.0, 84.0, 89.0, 96.0]
    MSinceMostRecentDelq = [18.0, 33.0, 48.0]
    MaxDelq2PublicRecLast12M = [6.0, 7.0]
    MaxDelqEver = [3.0]
    NumTotalTrades = [1.0, 10.0, 17.0, 28.0]
    NumTradesOpeninLast12M = [3.0, 4.0, 7.0, 12.0]
    PercentInstallTrades = [36.0, 47.0, 58.0, 85.0]
    MSinceMostRecentInqexcl7days = [1.0, 2.0, 9.0, 23.0]
    NumInqLast6M = [2.0, 5.0, 9.0]
    NumInqLast6Mexcl7days = [3.0]
    NetFractionRevolvingBurden = [15.0, 38.0, 73.0]
    NetFractionInstallBurden = [36.0, 71.0]
    NumRevolvingTradesWBalance = [4.0, 5.0, 8.0, 12.0]
    NumInstallTradesWBalance = [3.0, 4.0, 12.0, 14.0]
    NumBank2NatlTradesWHighUtilization = [2.0, 3.0, 4.0, 6.0]
    PercentTradesWBalance = [48.0, 67.0, 74.0, 87.0]

    # vect = Array{Int32,1}(undef, 23)
    # Mapping indices from Julia (1-based) to Python (0-based)
    # vect[1] (Julia) -> vect[0] (Python)
    # values[1] (Julia) -> values[0] (Python)
    vect = [0] * 23
    vect[0] = feature_to_vector(values[0], ExternalRiskEstimate)
    vect[1] = feature_to_vector(values[1], MSinceOldestTradeOpen)
    vect[2] = feature_to_vector(values[2], MSinceMostRecentTradeOpen)
    vect[3] = feature_to_vector(values[3], AverageMInFile)
    vect[4] = feature_to_vector(values[4], NumSatisfactoryTrades)
    vect[5] = feature_to_vector(values[5], NumTrades60Ever2DerogPubRec)
    vect[6] = feature_to_vector(values[6], NumTrades90Ever2DerogPubRec)
    vect[7] = feature_to_vector(values[11], NumTotalTrades) # values[12] -> values[11]
    vect[8] = feature_to_vector(values[12], NumTradesOpeninLast12M) # values[13] -> values[12]
    vect[9] = feature_to_vector(values[7], PercentTradesNeverDelq) # values[8] -> values[7]
    vect[10] = feature_to_vector(values[8], MSinceMostRecentDelq) # values[9] -> values[8]
    vect[11] = feature_to_vector(values[9], MaxDelq2PublicRecLast12M) # values[10] -> values[9]
    vect[12] = feature_to_vector(values[10], MaxDelqEver) # values[11] -> values[10]
    vect[13] = feature_to_vector(values[13], PercentInstallTrades) # values[14] -> values[13]
    vect[14] = feature_to_vector(values[18], NetFractionInstallBurden) # values[19] -> values[18]
    vect[15] = feature_to_vector(values[20], NumInstallTradesWBalance) # values[21] -> values[20]
    vect[16] = feature_to_vector(values[14], MSinceMostRecentInqexcl7days) # values[15] -> values[14]
    vect[17] = feature_to_vector(values[15], NumInqLast6M) # values[16] -> values[15]
    vect[18] = feature_to_vector(values[16], NumInqLast6Mexcl7days) # values[17] -> values[16]
    vect[19] = feature_to_vector(values[17], NetFractionRevolvingBurden) # values[18] -> values[17]
    vect[20] = feature_to_vector(values[19], NumRevolvingTradesWBalance) # values[20] -> values[19]
    vect[21] = feature_to_vector(values[21], NumBank2NatlTradesWHighUtilization) # values[22] -> values[21]
    vect[22] = feature_to_vector(values[22], PercentTradesWBalance) # values[23] -> values[22]

    return vect

## Subscore for each sublayer
def external_risk_subscore_from_vector(vect):
    weights = [2.9895622, 2.1651128, 1.4081029, 0.7686735, 0.0, 0.0, 0.0, 1.6943381]
    score = -1.4308699
    # vect[0] subtract 1 for Python indexing
    return score + weights[vect[0] - 1]

def trade_open_time_subscore_from_vector(vect):
    weights1 = [0.820842027, 0.525120503, 0.245257364, 0.005524848, 0.0, 0.418318111, 0.435851213]
    weights2 = [0.031074792, 0.006016629, 0.0, 0.0, 0.027688067]
    weights3 = [ 1.209930852, 0.694452470, 0.296029824, 0.0, 0.0, 0.0, 0.471490736]
    score = -0.696619002
    return score + weights1[vect[1] - 1] + weights2[vect[2] - 1] + weights3[vect[3] - 1]

def num_sat_trades_subscore_from_vector(vect):
    weights = [2.412574, 1.245278, 6.619963e-01, 2.731984e-01, 5.444148e-09, 0.0, 0.0, 4.338848e-01]
    score = -1.954726e-01
    return score + weights[vect[4] - 1]

def trade_freq_subscore_from_vector(vect):
    weights1 = [2.710260e-04, 9.195886e-01, 9.758620e-01, 1.008107e+01, 9.360290, 0.0, 0.0, 3.970360e-01]
    weights2 = [1.514937e-01, 3.139667e-01, 0.0, 2.422345e-01, 0.0, 0.0, 3.095043e-02]
    weights3 = [2.888436e-01, 9.659472e-01, 5.142479e-01, 2.653203e-01, 8.198233e-07, 0.0, 0.0, 3.233593e-01]
    weights4 =[8.405069e-06, 3.374686e-01, 4.934466e-01, 8.601860e-01, 9.451724, 0.0, 0.0, 1.351433e-01]
    score = -6.480598e-01
    return score + weights1[vect[5] - 1] + weights2[vect[6] - 1] + weights3[vect[7] - 1] + weights4[vect[8] - 1]

def delinquency_subscore_from_vector(vect):
    weights1 = [1.658975, 1.218405, 8.030501e-01, 5.685712e-01, 0.0, 0.0, 0.0, 6.645698e-01]
    weights2 = [4.014945e-01, 2.912651e-01, 5.665418e-02, 0.0, 6.935965e-01, 5.470874e-01, 4.786956e-01]
    weights3 = [1.004642, 5.654694e-01, 0.0, 0.0, 0.0, 2.841047e-01]
    weights4 = [1.378803e-01, 1.101649e-06, 0.0, 0.0, 1.051132e-02]
    score = -1.199469
    return score + weights1[vect[9] - 1] + weights2[vect[10] - 1] + weights3[vect[11] - 1] + weights4[vect[12] - 1]

def installment_subscore_from_vector(vect):
    weights1 = [9.059412e-05, 1.292266e-01, 4.680034e-01, 8.117938e-01, 1.954441, 0.0, 0.0, 1.281830]
    weights2 = [0.0, 1.432068e-01, 3.705526e-01, 0.0, 4.972869e-03, 1.513885e-01]
    weights3 = [1.489759, 1.478176, 1.518328, 0.0, 9.585058e-01, 0.0, 1.506442, 5.561296e-01]
    score = -1.750937
    return score + weights1[vect[13] - 1] + weights2[vect[14] - 1] + weights3[vect[15] - 1]

def inquiry_subscore_from_vector(vect):
    weights1 = [1.907737, 1.260966, 1.010585, 8.318137e-01, 0.0, 1.951357, 0.0, 1.719356]
    weights2 = [2.413596e-05, 2.251582e-01, 5.400251e-01, 1.255076, 0.0, 0.0, 1.061504e-01]
    weights3 = [0.0, 6.095516e-02, 0.0, 0.0, 1.125418e-02]
    score = -1.598351
    return score + weights1[vect[16] - 1] + weights2[vect[17] - 1] + weights3[vect[18] - 1]

def revol_balance_subscore_from_vector(vect):
    weights1 = [0.0001042232, 0.6764476961, 1.3938464180, 2.2581926077, 0.0, 1.7708134303, 1.0411847907]
    weights2 = [0.0756555085, 0.0, 0.1175915408, 0.2823307493, 0.4242649887, 0.0, 0.8756715032, 0.0897134843]
    score = -0.8924856930
    return score + weights1[vect[19] - 1] + weights2[vect[20] - 1]

def utilization_subscore_from_vector(vect):
    weights = [0.0, 0.8562096, 1.2047649, 1.1635459, 1.4701220, 0.0, 1.2392294, 0.4800086]
    score = -0.2415871
    return score + weights[vect[21] - 1]

def trade_w_balance_subscore_from_vector(vect):
    weights = [0.0, 0.5966752, 0.9207121, 1.2749998, 1.8474869, 0.0, 2.2885183, 1.0606029]
    score = -0.8221922
    return score + weights[vect[22] - 1]

# Sigmoid function
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-1.0 * x))

# the risk model to return the risks for that given entity
def risk_model(entity):
    vect = entity_to_vectors(entity)
    # subscore::Array{Float64,1} = Array{Float64,1}(undef, 10)
    subscore = [0.0] * 10
    subscore[0] = sigmoid(external_risk_subscore_from_vector(vect))
    subscore[1] = sigmoid(trade_open_time_subscore_from_vector(vect))
    subscore[2] = sigmoid(num_sat_trades_subscore_from_vector(vect))
    subscore[3] = sigmoid(trade_freq_subscore_from_vector(vect))
    subscore[4] = sigmoid(delinquency_subscore_from_vector(vect))
    subscore[5] = sigmoid(installment_subscore_from_vector(vect))
    subscore[6] = sigmoid(inquiry_subscore_from_vector(vect))
    subscore[7] = sigmoid(revol_balance_subscore_from_vector(vect))
    subscore[8] = sigmoid(utilization_subscore_from_vector(vect))
    subscore[9] = sigmoid(trade_w_balance_subscore_from_vector(vect))

    weights = [1.5671672, 2.5236825, 2.1711503, 0.3323177, 2.5396631, 0.9148520,
               3.0015073, 1.9259728, 0.9864329, 0.2949793]
    score = -8.3843046
    for i in range(len(weights)):
        score += (subscore[i] * weights[i])

    return sigmoid(score)



### GitHub Repository

In [ ]:
#!curl -fsSL https://julialang.org | sh -s -- -y
#import os
#os.environ['PATH'] += ":/root/.juliaup/bin"

In [ ]:
# 1. Install juliaup (Official version manager)
!curl -fsSL https://install.julialang.org | sh -s -- -y

# 2. Add juliaup to the system path
import os
os.environ['PATH'] += ":/root/.juliaup/bin"

# 3. Install and set Julia 1.6 as the default
!juliaup add 1.6
!juliaup default 1.6

# 4. Verify the version
!julia --version  # Should now say 1.6.x

In [ ]:
!ls /root/.juliaup/bin/

In [ ]:
!/root/.juliaup/bin/julia +1.6 --version

In [ ]:
!git clone https://github.com/GibbsG/GeneticCF
%cd GeneticCF

In [ ]:
%cd /content/GeneticCF/GeCo
!/root/.juliaup/bin/julia +1.6 --project=. -e 'using Pkg; Pkg.instantiate(); Pkg.status()'

In [ ]:
# Tell Julia to use Colab's Python for PyCall
!julia --project=. -e 'using Pkg; ENV["PYTHON"]=Sys.which("python"); Pkg.build("PyCall"); Pkg.precompile()'